# Name That LLM — enhanced 0.651 recipe

the input checker incorrectly
expected 32 layers for Qwen3-8B-Base. The mounted and official model has 36.
The check now requires 36, with independent architecture fixtures and an actual
preflight CLI regression. Inputs, prediction logic and weights are unchanged.
Official source: https://huggingface.co/Qwen/Qwen3-8B-Base/blob/main/config.json

This edits the supplied **0.651 Verified Reproduction**, rather than the earlier
0.643 notebook. The quoted 0.651 is the source author's reported baseline score;
**this enhancement has no measured leaderboard score yet**.

All **31 input references**, six main members, 18 v5 adapters, Gaussian/twin
heads, complementary stack and 10% ATLAS are retained. Neural forward functions,
precision, pooling, readouts and original ensemble weights remain unchanged.

The prediction experiment softly removes some of the existing lexical SVC's
inferred topic bias in groups of 8–18 responses. It uses the parent's text-only
grouping, confidence gating, and the SVC's original share in the blend. It adds
**zero GPU forward passes**. Sparse groups are unchanged. The original balancing,
typography rules and dense assignment remain; no new class quota is introduced.

## Measured evidence

Fresh train-only four-view SVC, **5 outer prompt folds × 3 inner folds**; vocabulary
and IDF refitted in every fold. Strength selected only on inner folds, averaging
complete, sparse and uneven-label scenarios. No public pool or released
task-trained adapter enters this experiment.

| Scenario | Baseline balanced accuracy | Candidate | Change |
|---|---:|---:|---:|
| full (2699 rows) | 0.38448 | 0.40191 | +0.01742 |
| sparse (373 rows) | 0.41289 | 0.41289 | +0.00000 |
| uneven (1546 rows) | 0.37823 | 0.39154 | +0.01331 |

These are **SVC proxy results before the full ensemble's postprocessing**, not
OOF results for the neural ensemble. Whole-prompt bootstrap intervals, split IDs,
predictions and reproducible validation source are in the accompanying package.
An improvement above public LB 0.651 remains unverified.

## Run on Kaggle

1. Import this notebook into a copy of the supplied reproduction and keep its
   **same 31 inputs**, including the verified base Llama cache.
2. Select **GPU T4 x2**, **Internet off**. Keep the first cell's defaults:
   `RUN_MODE="kaggle"`, `RUN_VALIDATION=False`.
3. **Save Version → Save & Run All**. The first cell prints a run marker;
   `INPUT PREFLIGHT: PASS` precedes any large model. This notebook performs real
   inference even for the three visible rows; there is no hardcoded preview.
4. Inspect `run_report.json`, `complement.log` and `main_inference.log`.
   `full_recipe_completed` must be true for a complete run. Submit the saved
   version that writes `/kaggle/working/submission.parquet`.

The source documentation reports **4 h 4 min** on the three visible rows on T4 x2.
That is not a hidden-test runtime forecast. Allow several hours; this version
enforces an **8.5-hour total deadline** and saves before optional ATLAS, reserving
time for final files. A skipped ATLAS is explicitly marked as a degraded run.
Required main/complement failures stop execution rather than silently dropping
models. Kaggle's outer 9-hour limit remains authoritative.

## What changed and how to review it

* Checked subprocesses retain actual errors and all logs; no `grep` pipeline
  masks a failed worker. Optional ATLAS runs separately with a hard timeout.
* Model discovery verifies base architecture and shard/tokenizer presence,
  follows linked mounts and collects missing inputs in one report. Unused old
  adapter families are no longer unnecessary startup dependencies.
* Zero-weight TF-IDF and unused family-gate fitting are skipped. Repeated
  in-sample checks are opt-in; checkpoint-anchor checks are retained.
* Exact normalized test-text overlaps in both active external pools are excluded and
  counted. This is not an authentication or near-duplicate audit of the public
  pools; upstream provenance remains a limitation.
* Both the selected candidate `submission.parquet` and a same-run baseline
  `submission_baseline_0651.parquet` are saved atomically with checked ID order.
  `main_evidence.npz` and `final_evidence.npz` retain reusable predictions.
  This baseline uses the same overlap filtering and any actual ATLAS completion;
  its filename does not confer a separately verified 0.651 score.

Set `TOPIC_ALPHA=0.0` to disable the sole new score correction. CPU fixtures
verify numerical baseline parity for grouping, final postprocessing and the
actual original-vs-enhanced train-only CLI. The full released neural assets were
not available for local GPU execution.

For an optional rerun of nested SVC validation, set `RUN_VALIDATION=True`; it
does not tune on test data or alter the selected constant automatically. For a
local plumbing check, use `RUN_MODE="cpu_smoke"`; it writes CPU-only predictions
and is not a competition ensemble submission. `preflight` checks inputs only.

Credits: original recipe and v5 adapters by **Dhruv Tibarewal**; supplied credited
reproduction by **avazaku**. Existing public-stack author comments are retained.
The soft SVC topic-correction idea follows the previously inspected credited
Roman Tamrazov public implementation, with contribution scaling and the exact
v27 grouping here. Attached models/data retain their own terms. **Built with Llama.**


In [ ]:
# Full inference is the default. No settings change is needed for Kaggle.
RUN_MODE = "kaggle"           # kaggle | preflight | cpu_smoke
TOPIC_ALPHA = 0.62           # 0.0 disables the new SVC correction
RUN_VALIDATION = False         # optional nested CPU proxy, not full-stack OOF
RUNTIME_LIMIT_HOURS = 8.5
RUN_IN_SAMPLE_CHECKS = False   # optional, repeated expensive seen-row checks

import os, sys, time
from pathlib import Path
RUN_STARTED = time.time()
# Environment overrides are for local reproducibility; Kaggle needs none.
RUN_MODE = os.environ.get("NTL_RUN_MODE", RUN_MODE)
DATA_DIR = Path(os.environ.get("NTL_DATA", "/kaggle/input/competitions/name-that-llm"))
if not (DATA_DIR / "train.csv").is_file() and Path("/kaggle/input/name-that-llm/train.csv").is_file():
    DATA_DIR = Path("/kaggle/input/name-that-llm")
OUTPUT_DIR = Path(os.environ.get("NTL_OUTPUT", "/kaggle/working")).resolve()
INPUT_DIR = Path(os.environ.get("NTL_INPUTS", "/kaggle/input"))
RUNTIME_DIR = OUTPUT_DIR / "ntl0651_runtime"
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
print("ENHANCED 0.651 NOTEBOOK START", {"mode": RUN_MODE, "topic_alpha": TOPIC_ALPHA,
      "run_validation": RUN_VALIDATION, "data": str(DATA_DIR), "output": str(OUTPUT_DIR)}, flush=True)


The following cells write the offline runtime. Run all cells in order; no additional package or helper upload is required.

In [ ]:
%%writefile "{RUNTIME_DIR}/ntl_upgrade.py"
"""Small offline additions to the supplied 0.651 recipe. No pretrained assets here."""
from pathlib import Path
import json, os, re, signal, subprocess, time, unicodedata, tempfile, shutil
import numpy as np
import pandas as pd
from scipy.special import log_softmax

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
FW = set('a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little'.split())
VIEWS = [('char',3,5,250000,1.),('word',1,3,180000,.7),('shape',2,6,60000,.5),('function',1,3,50000,.5)]
def shape(t): return re.sub(r'[^\W\d_]+',lambda m:'X' if m[0].isupper() else 'x',t)
def function(t): return ' '.join(w.lower() if w.lower() in FW or not w.isalnum() else 'CONTENT' for w in re.findall(r'\w+|[^\w\s]',t))

class FourViews:
    """Exactly the original main SVC's four transforms, vocabulary and coefficients."""
    def fit_transform(self,texts):
        from scipy.sparse import hstack
        from sklearn.feature_extraction.text import TfidfVectorizer
        self.views=[]; blocks=[]
        for view,lo,hi,mf,w in VIEWS:
            prep=shape if view=='shape' else function if view=='function' else str
            an='char' if view=='shape' else 'word' if view=='function' else view
            opts=dict(analyzer=an,ngram_range=(lo,hi),lowercase=False,sublinear_tf=True,min_df=2,max_features=mf,dtype=np.float32)
            if an=='word':opts['token_pattern']=r'(?u)\b\w+\b|[^\w\s]'
            vec=TfidfVectorizer(**opts);blocks.append(vec.fit_transform([prep(t) for t in texts])*w)
            self.views.append((vec,prep,w))
        return hstack(blocks,format='csr')
    def transform(self,texts):
        from scipy.sparse import hstack
        return hstack([vec.transform([prep(t) for t in texts])*w for vec,prep,w in self.views],format='csr')

def capped_agglo(S,cap=18,thr=.02):
    # Original v27 algorithm, including its deterministic tie breaking.
    n=len(S); A=S.astype(np.float32).copy();np.fill_diagonal(A,-np.inf)
    size=np.ones(n,np.int32);alive=np.ones(n,bool);lab=np.arange(n)
    while True:
        ok=(size[:,None]+size[None,:]<=cap)&alive[:,None]&alive[None,:]
        M=np.where(ok,A,-np.inf);k=int(np.argmax(M));i,j=divmod(k,n)
        if not np.isfinite(M[i,j]) or M[i,j]<thr:break
        new=(size[i]*A[i]+size[j]*A[j])/(size[i]+size[j])
        A[i]=new;A[:,i]=new;A[i,i]=-np.inf;A[j]=-np.inf;A[:,j]=-np.inf
        size[i]+=size[j];alive[j]=False;lab[lab==j]=i
    return lab

def topic_groups(texts,thr=.15):
    from sklearn.feature_extraction.text import TfidfVectorizer
    if len(texts)<2:return np.arange(len(texts))
    try:
        X=TfidfVectorizer(sublinear_tf=True,stop_words='english',min_df=2,max_df=.05).fit_transform(texts)
        return capped_agglo((X@X.T).toarray(),18,thr)
    except ValueError:return np.arange(len(texts))

def topic_correction(core,svc,groups,alpha,weight=1.):
    """Remove some SVC topic bias in inferred dense groups; no class assignment.

    The evidence change is scaled by the EXISTING SVC contribution to the main
    blend. Sparse groups, alpha=0, and absent SVC all give exact identity.
    """
    core=np.asarray(core);groups=np.asarray(groups)
    if alpha==0 or svc is None or weight==0:return core.copy()
    scores=np.asarray(svc,dtype=np.float64)
    if scores.shape!=core.shape or len(groups)!=len(core):raise ValueError('SVC evidence axes differ')
    centered=scores.copy();_,inverse,count=np.unique(groups,return_inverse=True,return_counts=True)
    eligible=(count[inverse]>=8)&(count[inverse]<=18)
    for g in np.unique(groups[eligible]):
        ix=groups==g;centered[ix]-=scores[ix].mean(0,keepdims=True)
    delta=log_softmax(centered,axis=1)-log_softmax(scores,axis=1);delta[~eligible]=0
    ordered=np.sort(core,axis=1);gate=np.clip((2-(ordered[:,-1]-ordered[:,-2]))/1.6,.15,1.)
    gate[~eligible]=0
    return core+float(alpha)*float(weight)*delta*gate[:,None]

def finite(name,x):
    x=np.asarray(x)
    if not np.isfinite(x).all():raise FloatingPointError(f'{name} contains NaN or infinity')
    return x

def write_submission(ids,pred,path):
    ids=np.asarray(ids,dtype=str);pred=np.asarray(pred,dtype=str)
    frame=pd.DataFrame({'id':ids,'model':pred})
    if len(frame)!=len(ids) or not frame.id.is_unique or not frame.model.isin(LABELS).all():
        raise ValueError('Invalid submission rows or labels')
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp.parquet')
    frame.to_parquet(tmp,index=False);check=pd.read_parquet(tmp)
    if not check.equals(frame):raise RuntimeError('Parquet readback mismatch')
    os.replace(tmp,path)
    return frame

def canonical(t):return re.sub(r'\s+',' ',unicodedata.normalize('NFKC',str(t))).strip().casefold()
def clean_pool(frame,test_texts):
    """Drop exact normalized test overlaps without ever reading test labels."""
    frame=frame[frame.model.isin(LABELS)].copy();frame['text']=frame.text.fillna('').astype(str)
    keys=frame.text.map(canonical);test_keys={canonical(t) for t in test_texts}
    keep=~keys.isin(test_keys)
    report={'eligible_rows':len(frame),'test_overlap_rows_removed':int((~keep).sum())}
    return frame[keep].reset_index(drop=True),report

def run_checked(argv,deadline,log_path):
    """Stream all output, retain actual exit status, kill descendants on timeout."""
    if deadline-time.time()<=0:raise TimeoutError('Notebook time budget exhausted')
    Path(log_path).parent.mkdir(parents=True,exist_ok=True)
    owns_registry='NTL_PROCESS_REGISTRY' not in os.environ
    registry=Path(os.environ.get('NTL_PROCESS_REGISTRY') or tempfile.mkdtemp(prefix='ntl-processes-',dir=Path(log_path).parent))
    env=dict(os.environ,NTL_PROCESS_REGISTRY=str(registry.resolve()))
    with open(log_path,'w') as log:
        proc=subprocess.Popen(list(map(str,argv)),stdout=log,stderr=subprocess.STDOUT,start_new_session=True,env=env)
        marker=registry/f'{proc.pid}.json';marker.write_text(json.dumps({'pid':proc.pid,'parent':os.getpid()}))
        offset=0
        try:
            while True:
                with open(log_path) as reader:
                    reader.seek(offset);chunk=reader.read();offset=reader.tell()
                if chunk:print(chunk,end='',flush=True)
                rc=proc.poll()
                if rc is not None:
                    with open(log_path) as reader:
                        reader.seek(offset);print(reader.read(),end='',flush=True)
                    if rc:raise subprocess.CalledProcessError(rc,argv)
                    return
                if time.time()>=deadline:raise TimeoutError(f'Time budget exhausted: {argv[1]}')
                time.sleep(.25)
        finally:
            if proc.poll() is None:
                # Nested supervised workers register their process groups in the
                # inherited registry. No privileged process-list command is needed.
                descendants=set()
                pairs=[]
                for entry in registry.glob('*.json'):
                    try:
                        info=json.loads(entry.read_text());pairs.append((info['pid'],info['parent']))
                    except (OSError,ValueError,KeyError):continue
                known={proc.pid}
                while True:
                    found={pid for pid,parent in pairs if parent in known}-known
                    if not found:break
                    known|=found;descendants|=found
                for pid in descendants:
                    try:os.killpg(pid,signal.SIGTERM)
                    except ProcessLookupError:pass
                try:os.killpg(proc.pid,signal.SIGTERM)
                except ProcessLookupError:pass
                try:proc.wait(timeout=1)
                except subprocess.TimeoutExpired:
                    try:os.killpg(proc.pid,signal.SIGKILL)
                    except ProcessLookupError:pass
                    proc.wait()
                for pid in descendants:
                    try:os.killpg(pid,signal.SIGKILL)
                    except ProcessLookupError:pass
            marker.unlink(missing_ok=True)
            if owns_registry:shutil.rmtree(registry,ignore_errors=True)

def final_scores(P,texts,groups,sinkhorn=1.,post=True,guards=()):
    """Original 100-iteration balancing, typography guards and dense assignment."""
    from scipy.optimize import linear_sum_assignment
    P=finite('blend probabilities',P);Q=P.copy()
    if sinkhorn>0 and len(P)>=18:
        for _ in range(100):
            Q*=(len(Q)/18)/Q.sum(0,keepdims=True);Q/=Q.sum(1,keepdims=True)
        total=(1-sinkhorn)*np.log(P+1e-12)+sinkhorn*np.log(Q+1e-12)
    else:total=np.log(P+1e-12)
    if post:
        for mark,allowed in guards:
            forbidden=np.array([l not in allowed for l in LABELS])
            for i,t in enumerate(texts):
                if mark in t and LABELS[int(total[i].argmax())] not in allowed:total[i,forbidden]=-1e9
        _,inv,count=np.unique(groups,return_inverse=True,return_counts=True)
        sizes=count[inv]
        for g in np.unique(np.asarray(groups)[(sizes>=12)&(sizes<=18)]):
            ix=np.flatnonzero(groups==g);r,c=linear_sum_assignment(-total[ix])
            total[ix[r]]=-1e9;total[ix[r],c]=0.
    return finite('final scores',total)


In [ ]:
%%writefile "{RUNTIME_DIR}/ntl_inputs.py"
"""Offline input preflight. Reads metadata only; never loads or downloads models."""
from pathlib import Path
import fnmatch, json, os, re

TARGETS={
 'qwen3_8b':dict(parts=('/qwen-3/','/8b-base/'),hints=('qwen38bbase',),kind='qwen3',hidden=4096,layers=36),
 'qwen3_14b':dict(parts=('/qwen-3/','/14b-base/'),hints=('qwen314bbase',),kind='qwen3',hidden=5120,layers=40),
 'llama8b':dict(parts=('/llama-3.1/','/8b/'),hints=('llama318b',),kind='llama',hidden=4096,layers=32),
 'qwen25_14b':dict(parts=('/qwen2.5/','/14b/'),hints=('qwen2514b',),kind='qwen2',hidden=5120,layers=48)}
LLAMA_SOURCE='/kaggle/input/models/metaresearch/llama-3.1/transformers/8b/2'

class InputPreflightError(RuntimeError):pass

def inventory(root):
    """Follow linked mounts, prune directory cycles, and deduplicate physical files."""
    root=Path(root);files=[];seen_dirs=set()
    if not root.is_dir():return files
    for base,dirs,names in os.walk(root,followlinks=True):
        real=os.path.realpath(base)
        if real in seen_dirs:dirs[:]=[];continue
        seen_dirs.add(real);dirs.sort()
        files.extend(Path(base)/n for n in sorted(names))
    return files

def unique(hits,label):
    physical={}
    for p in sorted(map(Path,hits),key=str):physical.setdefault(os.path.realpath(p),p)
    if len(physical)!=1:
        raise InputPreflightError(f'{label}: expected one distinct input, found {list(map(str,physical.values()))}')
    return next(iter(physical.values()))

def norm(s):return re.sub('[^a-z0-9]','',s.lower())

def model_config(path):
    d=json.loads(Path(path).read_text());return d.get('text_config') or d

def find_base(configs,key):
    spec=TARGETS[key];exact=[];other=[];rejected=[]
    for p in configs:
        try:c=model_config(p)
        except (ValueError,OSError):continue
        hint=(str(p)+' '+str(c.get('_name_or_path',''))).lower()
        named=all(part in str(p).lower() for part in spec['parts'])
        alias=any(x in norm(hint) for x in spec['hints'])
        if not (named or alias):continue
        if 'instruct' in hint:
            rejected.append(str(p)+' (instruction model is not the required base)');continue
        if c.get('quantization_config'):
            rejected.append(str(p)+' (quantized replacement is not the original fp16 base)');continue
        got=(c.get('model_type'),c.get('hidden_size'),c.get('num_hidden_layers'))
        expected=(spec['kind'],spec['hidden'],spec['layers'])
        if got!=expected:
            rejected.append(str(p)+f' (architecture {got}, expected {expected})');continue
        (exact if named else other).append(p)
    hits=exact or other
    if not hits:
        extra=f'; rejected: {rejected}' if rejected else ''
        source=f'; original v27 mounted {LLAMA_SOURCE}' if key=='llama8b' else ''
        raise InputPreflightError(f'{key}: compatible base model not found{source}{extra}')
    p=unique(hits,key).parent
    weights=list(p.glob('model*.safetensors'))+list(p.glob('pytorch_model*.bin'))
    if not weights:raise InputPreflightError(f'{key}: no model weight files in {p}')
    if not any((p/n).is_file() for n in ['tokenizer.json','tokenizer.model','vocab.json']):
        raise InputPreflightError(f'{key}: tokenizer files missing in {p}')
    for ix in list(p.glob('*.index.json')):
        try:required=set(json.loads(ix.read_text()).get('weight_map',{}).values())
        except (ValueError,OSError):continue
        missing=[n for n in required if not (p/n).is_file()]
        if missing:raise InputPreflightError(f'{key}: missing weight shards {missing}')
    return str(p)

def inspect_inputs(root):
    root=Path(root);files=inventory(root);configs=[p for p in files if p.name=='config.json']
    errors=[];warnings=[];found={}
    def get(label,fn):
        try:found[label]=fn();return found[label]
        except (InputPreflightError,ValueError,KeyError,OSError,TypeError,AssertionError) as e:
            errors.append({'input':label,'problem':str(e)});return None
    def match(pattern,label):return unique([p for p in files if fnmatch.fnmatch(str(p),pattern)],label)
    def adapter(name,head='heads.pt'):
        path=match(f'*/{name}/{head}',name).parent
        ad=path/'adapter' if (path/'adapter/adapter_config.json').is_file() else path
        for file in ['adapter_config.json','adapter_model.safetensors']:
            if not (ad/file).is_file():raise InputPreflightError(f'{name}: {ad/file} is missing')
        if not ((path/'heads.pt').is_file() or (path/'heads.safetensors').is_file()):
            raise InputPreflightError(f'{name}: trained heads missing')
        return str(path)
    def meta(name):
        fp=match(f'*/{name}/meta.json',name);c=json.loads(fp.read_text())['cfg']
        for k in ['keep','layers','max_len']:assert k in c,f'{name}: cfg.{k} is missing'
        head='heads.pt' if (fp.parent/'heads.pt').is_file() else 'heads.safetensors'
        adapter(name,head)
        return str(fp.parent),c
    bases={key:get(key,lambda key=key:find_base(configs,key)) for key in TARGETS}
    n14=get('n14b',lambda:meta('lora_qwen3_14b'));n25=get('q25_14',lambda:meta('lora_qwen25_14b'))
    def encoder():
        p=match('*/deberta_v3_large/meta.json','fine-tuned DeBERTa').parent
        required=['head.safetensors','encoder/config.json']
        missing=[n for n in required if not (p/n).is_file()]
        if not list((p/'encoder').glob('*.safetensors')):missing.append('encoder/*.safetensors')
        if missing:raise InputPreflightError(f'DeBERTa: missing {missing} in {p}')
        return str(p)
    deb=get('deb',encoder)
    families={}
    for name in ['q8v5','q14v5','ll8v5']:
        families[name]=[get(n,lambda n=n:adapter(n)) for n in [f'{name}_f{k}' for k in range(5)]+[name]]
    pool=get('public_pool',lambda:str(match('*/nlm_pool_v5.parquet','public SVC training pool')))
    L=[2,4,6,8,12,16,20,24];models=[]
    if deb:models.append(dict(name='deb',type='enc',ckpt=deb,w=1.))
    for name,key,value,w in [('n14b','qwen3_14b',n14,2.5),('q25_14','qwen25_14b',n25,1.)]:
        if value and bases[key]:
            ck,c=value;models.append(dict(name=name,type='l24',path=bases[key],adapters=[ck],keep=c['keep'],
                layers=c['layers'],w=w,bs=8,max_len=c['max_len'],twin=.35,no_final_norm=True,eos_id=c.get('eos_id')))
    for name,key,keep,w,bs,g in [('q8v5','qwen3_8b',24,2.,16,.44),('q14v5','qwen3_14b',27,3.,8,.70),('ll8v5','llama8b',24,1.,16,0.)]:
        if bases[key] and all(families[name]):
            models.append(dict(name=name,type='l24',path=bases[key],adapters=families[name],keep=keep,layers=L,
                               w=w,bs=bs,max_len=384,twin=.35,gauss=g))
    atlas=[p for p in files if p.name=='atlas_bundle.json'];atlas_dir=''
    if atlas:
        try:atlas_dir=str(unique(atlas,'ATLAS bundle').parent)
        except InputPreflightError as e:warnings.append(str(e))
    else:warnings.append('ATLAS bundle not found; optional ATLAS contribution would be skipped')
    # Check auxiliary assets separately; do not replace their original worker discovery logic.
    comp_markers=['*glm4_9b_trunc25/config.json','*deberta-v3-large/config.json','*electra-large-discriminator/config.json',
                  '*desklib-detector/config.json','*tokattn_debl23_final.pt','*ultrabert-380m-ntl/fold0/config.json']
    for pattern in comp_markers:
        if not any(fnmatch.fnmatch(str(p),pattern) for p in files):warnings.append('Complement input not visible: '+pattern)
    for typ,h in [('glm4',4096),('qwen2',3584),('qwen3',2560)]:
        matches=[]
        for p in configs:
            try:c=model_config(p)
            except (ValueError,OSError):continue
            if c.get('model_type')==typ and c.get('hidden_size')==h:matches.append(p)
        if not matches:warnings.append(f'Complement backbone not visible: model_type={typ}, hidden_size={h}')
    rows=[]
    for p in configs:
        try:
            c=model_config(p);rows.append(dict(path=str(p),model_type=c.get('model_type'),hidden_size=c.get('hidden_size'),
                num_hidden_layers=c.get('num_hidden_layers'),name_or_path=c.get('_name_or_path')))
        except (ValueError,OSError):pass
    return dict(ok=not errors,input_root=str(root),errors=errors,warnings=warnings,config_inventory=rows,
                original_llama_source=LLAMA_SOURCE,manifest=dict(models=sorted(models,key=lambda s:['q14v5','q8v5','ll8v5','n14b','q25_14','deb'].index(s['name'])),pool=pool,atlas=atlas_dir,atlas_base=bases['qwen3_8b']))

def preflight(root,report_path,strict=True):
    report=inspect_inputs(root);path=Path(report_path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(report,indent=2))
    print(f'INPUT PREFLIGHT: {"PASS" if report["ok"] else "FAIL"}; report: {path}',flush=True)
    for err in report['errors']:print('MISSING/INVALID:',err['input'],'—',err['problem'],flush=True)
    for msg in report['warnings']:print('OPTIONAL:',msg,flush=True)
    if strict and not report['ok']:
        raise InputPreflightError(f'{len(report["errors"])} required inputs unavailable. '
            'See preflight_report.json for all problems; no large model was started. Restore the inputs from the scored v27 notebook.')
    return report


In [ ]:
%%writefile "{RUNTIME_DIR}/gauss.py"
"""Shared-prompt Gaussian group head (ported, Apache-2.0, from public Kaggle notebooks by prvsiyan / kirilltym / sangam21).
Two-way model x = class mean + prompt effect + noise; exact (n<=3) or cavity-EM (n>3) injective group marginals."""
import itertools
import numpy as np
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out


In [ ]:
%%writefile "{RUNTIME_DIR}/infer.py"
"""End-to-end attribution pipeline (self-contained; runs on Kaggle T4x2 or locally in --sim mode).
1. Mean-pooled hidden states from frozen LMs for train+test (computed here so precision/quantisation match).
2. Per model: LR on raw + LR on prompt-centred features (train centred by prompt_id, test by estimated cluster).
3. TF-IDF (char_wb + word/punct) prompt-centred LR.
4. Test prompts recovered by size-capped average-linkage clustering on content-word TF-IDF.
5. Sum log-probs, then Hungarian assignment inside each cluster (each model answers each prompt at most once).
"""
import os, sys, time, gc, argparse, json, signal
def _terminate(signum,frame):raise SystemExit('Supervisor requested termination')
signal.signal(signal.SIGTERM,_terminate)
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',TOKENIZERS_PARALLELISM='false')
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np, pandas as pd, scipy.sparse as sp, torch
from scipy.optimize import linear_sum_assignment
from pathlib import Path
from ntl_upgrade import FourViews,topic_correction,finite,write_submission,clean_pool,run_checked,final_scores
from scipy.special import softmax,log_softmax

LABELS = ['claude-fable-5','claude-opus-4-6','claude-opus-4-8','claude-opus-5','claude-sonnet-5',
 'gemini-3.1-pro-preview','gemini-3.6-flash','gemini-3.7-flash','gemini-3.8-flash','glm-5.3','glm-5.3-flash',
 'gpt-5.5','gpt-5.6-luna','gpt-5.6-sol','gpt-5.6-terra','gpt-6-astra','kimi-k3','qwen3.8-max']
L2I = {l: i for i, l in enumerate(LABELS)}

p = argparse.ArgumentParser()
p.add_argument('--data', default='/kaggle/input/competitions/name-that-llm')
p.add_argument('--models', default='')   # json list of {"path":..,"layers":[fractions],"quant":"fp16|4bit|8bit","bs":16}
p.add_argument('--comp_path', default=''); p.add_argument('--comp_w', type=float, default=0.0)
p.add_argument('--fake_test', type=int, default=0, help='timing probe: score N pool passages as a stand-in test')
p.add_argument('--atlas_probe', type=int, default=0, help='timing probe: stop ATLAS after N passages')
p.add_argument('--sim', type=str, default='', help='comma list of folds held out of train.csv as fake test')
p.add_argument('--out', default='submission.parquet')
p.add_argument('--cache', default='cache')
p.add_argument('--dtype', default='fp16')
p.add_argument('--C', type=float, default=0.005)
p.add_argument('--tfidf_w', type=float, default=1.0)
p.add_argument('--distort_w', type=float, default=0.0, help='weight of topic-masked (text distortion) TF-IDF member')
p.add_argument('--adapt_thr', type=float, default=0.15, help='cluster cut (avg cosine sim on content-word TF-IDF)')
p.add_argument('--adapt_w', type=float, default=0.0, help='weight of centred twin evidence on rows in clusters of size>=2 (0=off)')
p.add_argument('--post', type=int, default=0, help='1: spaced em-dash guard + Hungarian on dense test clusters (12-18)')
p.add_argument('--atlas_dir', default=''); p.add_argument('--atlas_base', default=''); p.add_argument('--atlas_w', default='0', help='mix weight per bundle (comma list or one value)')
p.add_argument('--atlas_margin', type=float, default=0.0, help='>0: score ATLAS only on rows whose top-2 margin is below this')
p.add_argument('--atlas_bs', type=int, default=4); p.add_argument('--atlas_chunk', type=int, default=3); p.add_argument('--atlas_budget', type=float, default=8.3 * 3600)
p.add_argument('--sk_div', type=int, default=1, help='1: divide blend by #members before Sinkhorn (ours); 0: raw blend (public notebook)')
p.add_argument('--sinkhorn', type=float, default=0.0, help='alpha of Sinkhorn class-marginal balancing (0=off)')
p.add_argument('--sim_k', default='', help="sim only: keep 'lo-hi' random passages per held-out prompt")
p.add_argument('--extra', default='', help='parquet/csv (text, model) of matched external passages, used by TF-IDF/distort members')
p.add_argument('--extra_w', type=float, default=0.5)
p.add_argument('--svc_w', type=float, default=0.0, help='4-view n-gram LinearSVC on train + --pool')
p.add_argument('--pool', default='')
p.add_argument('--save_lp', default='', help='also save normalised blend log-probs (n x 18, LABELS order) to this .npy')
p.add_argument('--perpassage', type=int, default=1, help='1: test passages treated independently (no clustering/centring/assignment)')
p.add_argument('--topic_alpha',type=float,default=0.)
a = p.parse_args()
if a.sim and any([json.loads(a.models or '[]'),a.extra,a.pool,a.atlas_dir,a.comp_path]):
    raise ValueError('Released task-trained checkpoints and external pools are not prompt-held-out validation. Run validate_upgrade.py.')
if not 0<=a.topic_alpha<=1:raise ValueError('topic_alpha must be in [0,1]')
if not a.perpassage:raise ValueError('This enhancement supports the scored per-passage recipe only')
os.makedirs(a.cache, exist_ok=True)
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False   # fp32 sums inside fp16 matmuls (free on T4)
T0 = time.time()
def log(*x): print(f'[{time.time()-T0:7.0f}s]', *x, flush=True)

# ---------------- data
train = pd.read_csv(os.path.join(a.data, 'train.csv'),dtype={'id':str,'prompt_id':str})
if a.sim:
    rng = np.random.RandomState(0); pids = np.array(sorted(train.prompt_id.unique())); rng.shuffle(pids)
    pf = {q: i % 5 for i, q in enumerate(pids)}; fo = train.prompt_id.map(pf).values
    hold = np.isin(fo, [int(x) for x in a.sim.split(',')])
    if a.sim_k:
        lo_, hi_ = map(int, a.sim_k.split('-')); keep = np.zeros(len(train), bool); idx_h = np.where(hold)[0]
        for g in np.unique(train.prompt_id.values[idx_h]):
            ii = idx_h[train.prompt_id.values[idx_h] == g]; keep[rng.choice(ii, min(len(ii), rng.randint(lo_, hi_ + 1)), replace=False)] = True
        hold = keep | (hold & False)
    test = train[hold].drop(columns=['model']).reset_index(drop=True); test_truth = train[hold].reset_index(drop=True)
    hold_any = np.isin(fo, [int(x) for x in a.sim.split(',')])
    train = train[~hold_any].reset_index(drop=True)
elif a.fake_test:
    test = pd.read_parquet(a.pool).sample(a.fake_test, random_state=0)[['text']].reset_index(drop=True)
    test['id'] = [f'fake{i}' for i in range(len(test))]
else:
    test = pd.read_csv(os.path.join(a.data, 'test.csv'),dtype={'id':str})
train['text'] = train.text.fillna(''); test['text'] = test.text.fillna('')
y = train.model.map(L2I).values; Gtr = train.prompt_id.values
log('train', len(train), 'test', len(test))

# ---------------- utils
def center_groups(X, groups):
    X = X.copy()
    for g in np.unique(groups):
        i = groups == g; X[i] -= X[i].mean(0)
    return X

def torch_lr(Xtr, ytr, Xte, C, iters=200):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    Xtr = torch.as_tensor(Xtr, dtype=torch.float32, device=dev); Xte = torch.as_tensor(Xte, dtype=torch.float32, device=dev)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6; Xtr = (Xtr - mu) / sd; Xte = (Xte - mu) / sd
    yt = torch.as_tensor(ytr, dtype=torch.long, device=dev)
    W = torch.zeros(Xtr.shape[1], 18, device=dev, requires_grad=True); b = torch.zeros(18, device=dev, requires_grad=True)
    opt = torch.optim.LBFGS([W, b], lr=1, max_iter=iters, history_size=20, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(Xtr @ W + b, yt, reduction='sum') + 0.5 / C * (W ** 2).sum()
        loss.backward(); return loss
    opt.step(closure)
    with torch.no_grad(): return torch.log_softmax(Xte @ W + b, -1).cpu().numpy()

def capped_agglo(S, cap=18, thr=0.02):
    """Average linkage (Lance-Williams), never merging past `cap` members; stop when best avg sim < thr."""
    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)
    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)
    while True:
        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]
        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)
        if not np.isfinite(M[i, j]) or M[i, j] < thr: break
        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])
        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf
        size[i] += size[j]; alive[j] = False; lab[lab == j] = i
    return lab

def hungarian(lp, groups):
    pred = lp.argmax(1).copy()
    for g in np.unique(groups):
        idx = np.where(groups == g)[0]; r, c = linear_sum_assignment(-lp[idx]); pred[idx[r]] = c
    return pred

# ---------------- 1. recover test prompts
from sklearn.feature_extraction.text import TfidfVectorizer
if a.perpassage and a.adapt_w <= 0:
    Gte = np.arange(len(test))
else:
  try:
    v = TfidfVectorizer(sublinear_tf=True, stop_words='english', min_df=2, max_df=0.05)
    Xc = v.fit_transform(test.text.values)
    Gte = capped_agglo((Xc @ Xc.T).toarray(), 18, a.adapt_thr if a.perpassage else 0.02)
  except ValueError as e:  # tiny interactive test file: nothing survives df pruning
    log('clustering fallback:', e); Gte = np.arange(len(test))
log('test clusters', len(np.unique(Gte)), 'sizes', np.bincount(np.bincount(Gte)[np.bincount(Gte) > 0]).tolist())
if a.sim:
    from sklearn.metrics import adjusted_rand_score
    log('cluster ARI vs truth', round(adjusted_rand_score(test_truth.prompt_id, Gte), 4))

LP = {}
EXT_T, EXT_Y = np.array([], dtype=object), np.array([], dtype=int)
if a.extra and (a.tfidf_w>0 or a.distort_w>0):
    _E = pd.read_parquet(a.extra) if a.extra.endswith('.parquet') else pd.read_csv(a.extra)
    _E = _E[_E.model.isin(LABELS)]; EXT_T = _E.text.astype(str).values; EXT_Y = _E.model.map(L2I).values
    log('extra rows', len(EXT_T))
FIT_T = np.concatenate([train.text.values, EXT_T]); FIT_Y = np.concatenate([y, EXT_Y])
FIT_W = np.concatenate([np.ones(len(train)), np.full(len(EXT_T), a.extra_w)])
# ---------------- 2. TF-IDF centred LR
from sklearn.linear_model import LogisticRegression
if a.tfidf_w>0:
    mats_tr, mats_te = [], []
    for kw in [dict(analyzer='char_wb', ngram_range=(2, 5), min_df=3, max_features=60000),
               dict(token_pattern=r"\w+|[^\w\s]", ngram_range=(1, 2), min_df=3, max_features=30000)]:
        vec = TfidfVectorizer(sublinear_tf=True, lowercase=False, **kw).fit(FIT_T if a.perpassage else train.text.values)
        mats_tr.append(vec.transform(FIT_T if a.perpassage else train.text.values)); mats_te.append(vec.transform(test.text.values))
    if a.perpassage:
        LP['tfidf'] = LogisticRegression(C=20, max_iter=3000).fit(sp.hstack(mats_tr).tocsr(), FIT_Y, sample_weight=FIT_W).predict_log_proba(sp.hstack(mats_te).tocsr()) * a.tfidf_w
    else:
        Xtr = center_groups(sp.hstack(mats_tr).toarray().astype(np.float32), Gtr)
        Xte = center_groups(sp.hstack(mats_te).toarray().astype(np.float32), Gte)
        LP['tfidf'] = LogisticRegression(C=5, max_iter=3000).fit(Xtr, y).predict_log_proba(Xte) * a.tfidf_w
        del Xtr, Xte
    gc.collect(); log('tfidf done')
SVC_SCORES=None
POOL_AUDIT={}
if a.svc_w>0:
    from sklearn.svm import LinearSVC
    _P=pd.read_parquet(a.pool) if a.pool else pd.DataFrame(columns=['text','model'])
    _P,POOL_AUDIT=clean_pool(_P,test.text.values)
    _fit=list(train.text.values)+list(_P.text.values);_fy=np.r_[y,_P.model.map(L2I).values].astype(int)
    _views=FourViews();_A=_views.fit_transform(_fit);_B=_views.transform(test.text.tolist())
    _svc=LinearSVC(C=.5,random_state=42,max_iter=5000,class_weight='balanced').fit(_A,_fy)
    assert np.array_equal(_svc.classes_,np.arange(18))
    SVC_SCORES=_svc.decision_function(_B)
    LP['svc4']=log_softmax(SVC_SCORES,axis=1)*a.svc_w
    del _A,_B,_views,_svc;gc.collect();log('svc4 done',POOL_AUDIT)
if a.distort_w > 0:  # Stamatatos-style text distortion: keep top-100 frequent words + punctuation, mask the rest
    import re, collections
    wre = re.compile(r"[A-Za-z’']+")
    cnt = collections.Counter(w.lower() for t in FIT_T for w in wre.findall(t))
    keep = {w for w, _ in cnt.most_common(100)}
    dist = lambda arr: [wre.sub(lambda m_: m_.group(0) if m_.group(0).lower() in keep else '*', t) for t in arr]
    Dtr, Dte = dist(FIT_T), dist(test.text.values)
    vs = [TfidfVectorizer(sublinear_tf=True, lowercase=False, analyzer='char', ngram_range=(2, 6), min_df=2),
          TfidfVectorizer(sublinear_tf=True, lowercase=True, token_pattern=r"[^\s]+", ngram_range=(1, 3), min_df=2)]
    Xa = sp.hstack([v_.fit_transform(Dtr) for v_ in vs]).tocsr(); Xb = sp.hstack([v_.transform(Dte) for v_ in vs]).tocsr()
    LP['distort'] = LogisticRegression(C=10, max_iter=3000).fit(Xa, FIT_Y, sample_weight=FIT_W).predict_log_proba(Xb) * a.distort_w
    log('distort done')

# ---------------- 3. LM embeddings
if json.loads(a.models or "[]"):
    from transformers import AutoTokenizer, AutoModel
@torch.no_grad()
def embed(spec, texts):
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    kw = dict(device_map='auto')
    q = spec.get('quant', 'fp16')
    if q == '4bit':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
    elif q == '8bit':
        from transformers import BitsAndBytesConfig
        kw['quantization_config'] = BitsAndBytesConfig(load_in_8bit=True)
    kw['dtype'] = torch.bfloat16 if q == 'bf16' else torch.float16
    m = AutoModel.from_pretrained(spec['path'], **kw).eval()
    enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(x) for x in enc]); out = None; bs = spec.get('bs', 16); nan_rows = 0
    first = next(m.parameters()).device
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)
        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True).hidden_states
        nL = len(hs) - 1; sel = spec['layers_abs'] if 'layers_abs' in spec else [max(1, round(f * nL)) for f in spec['layers']]
        pooled = []
        for l in sel:
            h = hs[l].float(); w = att.to(h.device).unsqueeze(-1).float()
            pooled.append(((h * w).sum(1) / w.sum(1)).cpu())
        pooled = torch.cat(pooled, 1).numpy()
        nan_rows += int((~np.isfinite(pooled)).any(1).sum())
        if out is None: out = np.zeros((len(texts), pooled.shape[1]), np.float32); log('  layers', sel, 'of', nL)
        out[b] = pooled
    del m; gc.collect(); torch.cuda.empty_cache()
    if nan_rows: log('  WARNING non-finite rows:', nan_rows)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

TPL = 'Identify which AI language model wrote the following passage, based only on its writing style.\n\nPassage:\n{}\n\nThe author model is:'
@torch.no_grad()
def lora_predict(spec, texts):
    """LoRA classifier WITHOUT peft: merge W += (B @ A) * alpha/r into the base weights, load the trained score head."""
    from transformers import AutoModelForSequenceClassification
    from safetensors.torch import load_file
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    m = AutoModelForSequenceClassification.from_pretrained(spec['path'], num_labels=18, dtype=torch.float16, device_map='auto')
    m.config.pad_token_id = tok.pad_token_id
    cfg = json.load(open(os.path.join(spec['adapter'], 'adapter_config.json')))
    scale = cfg['lora_alpha'] / cfg['r']
    sd = load_file(os.path.join(spec['adapter'], 'adapter_model.safetensors'))
    mods = dict(m.named_modules()); n_merged = 0; head = None
    for k, v in sd.items():
        key = k.replace('base_model.model.', '', 1)
        if '.lora_A.' in key:
            mod_name = key.split('.lora_A.')[0]; B = sd[k.replace('.lora_A.', '.lora_B.')]
            lin = mods[mod_name]; W = lin.weight
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype); n_merged += 1
        elif key.startswith('score.') and 'weight' in key and 'original_module' not in key:
            head = v
    assert head is not None and n_merged > 0, (n_merged, [k for k in sd if 'score' in k])
    m.score.weight.copy_(head.to(m.score.weight.device, m.score.weight.dtype))
    log(f'  merged {n_merged} LoRA pairs (scale {scale}), head {tuple(head.shape)}')
    del mods, sd, lin, W, B, head; gc.collect()  # drop every reference to the model's modules
    m.eval()
    enc = tok(list(texts), truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(x) for x in enc]); out = np.zeros((len(texts), 18), np.float32); bs = spec.get('bs', 16)
    first = next(m.parameters()).device
    for i in range(0, len(order), bs):
        b = order[i:i + bs]; L = max(len(enc[j]) for j in b)
        ids = torch.full((len(b), L), tok.pad_token_id); att = torch.zeros((len(b), L), dtype=torch.long)
        for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
        lo = m(input_ids=ids.to(first), attention_mask=att.to(first)).logits.float()
        out[b] = torch.log_softmax(lo, -1).cpu().numpy()
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

@torch.no_grad()
def l24_predict(spec, texts, groups=None, train_texts=None):
    """LoRA24-recipe member(s): truncated base loaded ONCE; for each adapter dir in spec['adapters'] (or spec['adapter'])
    merge W += B A * s, compute masked mean-pooled multi-layer features, read plain (+ centred) heads, then unmerge.
    Returns log-probs averaged over adapters (plain, centred-or-None)."""
    from transformers import AutoConfig
    from safetensors.torch import load_file
    adapters = spec.get('adapters') or [spec['adapter']]
    tok = AutoTokenizer.from_pretrained(spec['path'])
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    cfg = AutoConfig.from_pretrained(spec['path']); tc = cfg.text_config if hasattr(cfg, 'text_config') else cfg
    tc.num_hidden_layers = spec['keep']; tc.use_cache = False
    if getattr(tc, 'layer_types', None) is not None: tc.layer_types = tc.layer_types[:spec['keep']]
    m = AutoModel.from_pretrained(spec['path'], config=cfg, dtype=torch.float16, device_map='auto').eval()
    if spec.get('no_final_norm'): m.norm = torch.nn.Identity()
    mods = dict(m.named_modules()); first = next(m.parameters()).device; L = spec['layers']
    _eos = spec.get('eos_id')   # checkpoints trained with an end token appended (Nishant's) need the same id at inference
    _tk = lambda tx: [x + ([_eos] if _eos is not None else []) for x in tok(list(tx), truncation=True, max_length=spec.get('max_len', 384) - (1 if _eos is not None else 0))['input_ids']]
    enc = _tk(texts)
    order = np.argsort([len(x) for x in enc]); bs = spec.get('bs', 16)
    def lsm(z): return z - z.max(1, keepdims=True) - np.log(np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True))
    P_acc, C_acc = 0.0, 0.0; FEATS = None; OVF = [0]
    if train_texts is not None:
        enc_tr = _tk(train_texts)
    def featurize(enc_, norms, hid):
        order_ = np.argsort([len(x) for x in enc_]); Z_ = np.zeros((len(enc_), hid * len(L)), np.float32)
        for i in range(0, len(order_), bs):
            b = order_[i:i + bs]; Lm = max(len(enc_[j]) for j in b)
            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
            for r, j in enumerate(b): ids[r, :len(enc_[j])] = torch.tensor(enc_[j]); att[r, :len(enc_[j])] = 1
            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in L):
                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
            Z_[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
        return Z_
    for ai, ad in enumerate(adapters):
        adp = os.path.join(ad, 'adapter') if os.path.exists(os.path.join(ad, 'adapter', 'adapter_config.json')) else ad
        acfg = json.load(open(os.path.join(adp, 'adapter_config.json'))); scale = acfg['lora_alpha'] / acfg['r']
        sd = load_file(os.path.join(adp, 'adapter_model.safetensors')); deltas = []
        for k, v in sd.items():
            if '.lora_A.' not in k: continue
            name = k.replace('base_model.model.', '', 1).split('.lora_A.')[0]
            if name not in mods and ('model.' + name) in mods: name = 'model.' + name
            W = mods[name].weight; B = sd[k.replace('.lora_A.', '.lora_B.')]
            W += ((B.to(W.device, torch.float32) @ v.to(W.device, torch.float32)) * scale).to(W.dtype)
            deltas.append((W, B, v))   # keep only the low-rank factors; the dense delta is recomputed to unmerge
        if os.path.exists(os.path.join(ad, 'heads.pt')): H = torch.load(os.path.join(ad, 'heads.pt'), map_location='cpu')
        else: H = dict(load_file(os.path.join(ad, 'heads.safetensors')))
        if os.path.exists(os.path.join(ad, 'meta.json')):   # remap head rows from the checkpoint's label order to ours
            _lab = json.load(open(os.path.join(ad, 'meta.json')))['labels']; _perm = [_lab.index(l) for l in LABELS]
            for _p in ('plain', 'centered'): H[f'{_p}.weight'] = H[f'{_p}.weight'][_perm]; H[f'{_p}.bias'] = H[f'{_p}.bias'][_perm]
        hid = H['norms.0.weight'].shape[0]
        norms = [torch.nn.LayerNorm(hid) for _ in L]
        for i, nm in enumerate(norms): nm.load_state_dict({'weight': H[f'norms.{i}.weight'], 'bias': H[f'norms.{i}.bias']})
        Z = np.zeros((len(texts), hid * len(L)), np.float32)
        for i in range(0, len(order), bs):
            b = order[i:i + bs]; Lm = max(len(enc[j]) for j in b)
            ids = torch.full((len(b), Lm), tok.pad_token_id); att = torch.zeros((len(b), Lm), dtype=torch.long)
            for r, j in enumerate(b): ids[r, :len(enc[j])] = torch.tensor(enc[j]); att[r, :len(enc[j])] = 1
            hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in L):   # fp16 overflow (un-normed deep states): redo this batch in bf16
                m.to(torch.bfloat16); hs = m(input_ids=ids.to(first), attention_mask=att.to(first), output_hidden_states=True, use_cache=False).hidden_states; m.to(torch.float16); OVF[0] += 1
            mask = att.unsqueeze(-1).float(); mask[:, 0] = 0
            Z[b] = torch.cat([nm((hs[l].float().cpu() * mask).sum(1) / mask.sum(1)) for l, nm in zip(L, norms)], -1).numpy()
        if train_texts is not None and ai == len(adapters) - 1:   # full-data adapter: features for the Gaussian head
            FEATS = (Z.copy(), featurize(enc_tr, norms, hid))
        for W, B, A_ in deltas:   # unmerge before the next adapter
            W -= ((B.to(W.device, torch.float32) @ A_.to(W.device, torch.float32)) * scale).to(W.dtype)
        torch.cuda.empty_cache()
        del deltas, sd; gc.collect()
        hd = lambda p, X: lsm(X @ H[f'{p}.weight'].float().numpy().T + H[f'{p}.bias'].float().numpy())
        P_acc = P_acc + hd('plain', Z) / len(adapters)
        if groups is not None:
            Zc = Z.copy()
            for g in np.unique(groups):
                ii = groups == g; Zc[ii] -= Zc[ii].mean(0)
            C_acc = C_acc + hd('centered', Zc) / len(adapters)
        log(f'  l24 {spec["name"]}: adapter {os.path.basename(ad)} done (bf16 redo batches {OVF[0]})')
    del m, mods; gc.collect(); torch.cuda.empty_cache()
    if train_texts is not None: return P_acc, (C_acc if groups is not None else None), FEATS
    return P_acc, (C_acc if groups is not None else None)

def gpu_mem():
    return ' '.join(f'gpu{i}:{torch.cuda.memory_allocated(i)/1e9:.1f}GB' for i in range(torch.cuda.device_count()))

specs = json.loads(a.models) if a.models else []
all_text = np.concatenate([train.text.values, test.text.values])
for spec in specs:
    if spec.get('type') == 'l24':
        log('l24', spec['name'])
        if len(test) < 50 and os.environ.get('NTL_PARITY_CHECK')=='1':
            _p40, _ = l24_predict(spec, train.text.values[:40]); _acc = float((_p40.argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, f'{spec["name"]} l24 does not reproduce its fit'
        if spec.get('gauss', 0) > 0:
            plain, cent, (Zte_g, Ztr_g) = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None, train_texts=train.text.values)
            import gauss as _ga
            prm = _ga.fit(Ztr_g, y, Gtr, dict(red='wpca', d=128, se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0))
            prm['T1'] = prm['Tg'] = 3.0
            _sz = np.bincount(Gte)[Gte]; _rows = (_sz >= 2) & (_sz <= 18)
            _q = _ga.scores(Ztr_g[[0, 1]], np.zeros(2, dtype=int), prm); assert np.allclose(np.exp(_q).sum(1), 1)
            _G = np.zeros((len(test), 18))
            if _rows.any():
                _G[_rows] = _ga.scores(Zte_g[_rows], np.unique(Gte[_rows], return_inverse=True)[1], prm)
            LP[spec['name'] + '_gauss'] = _G * spec['gauss']; log(f"  gaussian head {spec['name']}: {int(_rows.sum())} grouped test rows, weight {spec['gauss']}")
        else:
            plain, cent = l24_predict(spec, test.text.values, Gte if a.adapt_w > 0 else None)
        LP[spec['name']] = plain * spec.get('w', 1.0)
        if cent is not None:
            rows = np.bincount(Gte)[Gte] >= 2
            LP[spec['name'] + '_twin'] = np.where(rows[:, None], cent, 0.0) * a.adapt_w * spec.get('w', 1.0) * spec.get('twin', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'enc':   # fully fine-tuned encoder (DeBERTa-style): fp32, mean of the last layer, linear head
        log('enc', spec['name']); from safetensors.torch import load_file as _lf
        _ck = spec['ckpt']; _meta = json.load(open(os.path.join(_ck, 'meta.json')))
        _perm = [_meta['labels'].index(l) for l in LABELS]
        _hd = _lf(os.path.join(_ck, 'head.safetensors')); _W = _hd['weight'].float().numpy()[_perm]; _b = _hd['bias'].float().numpy()[_perm]
        _etok = AutoTokenizer.from_pretrained(os.path.join(_ck, 'encoder'))
        _enc = AutoModel.from_pretrained(os.path.join(_ck, 'encoder'), dtype=torch.float32).to('cuda:0').eval()
        def _enc_feats(texts):
            _ids = _etok(list(texts), truncation=True, max_length=_meta['max_len'])['input_ids']
            _ord = np.argsort([len(x) for x in _ids], kind='stable'); Z_ = np.zeros((len(_ids), _enc.config.hidden_size), np.float32)
            with torch.inference_mode():
                for _s in range(0, len(_ord), 32):
                    _idx = _ord[_s:_s + 32]; _bt = _etok.pad({'input_ids': [_ids[i] for i in _idx]}, return_tensors='pt').to('cuda:0')
                    _m = _bt['attention_mask'].unsqueeze(-1).float()
                    Z_[_idx] = ((_enc(**_bt).last_hidden_state * _m).sum(1) / _m.sum(1).clamp(min=1)).cpu().numpy()
            return Z_
        if len(test) < 50 and os.environ.get('NTL_PARITY_CHECK')=='1':
            _acc = float(((_enc_feats(train.text.values[:40]) @ _W.T + _b).argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, spec['name']
        _zz = _enc_feats(test.text.values) @ _W.T + _b
        LP[spec['name']] = (_zz - _zz.max(1, keepdims=True) - np.log(np.exp(_zz - _zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)
        del _enc; gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'head':
        log('head', spec['name']); H = np.load(spec['head'])
        Ete = embed(dict(spec, layers=sorted(float(v) for v in H['layers'])), test.text.values)  # extraction stored layers in sorted order
        if len(test) < 50 and os.environ.get('NTL_PARITY_CHECK')=='1':  # interactive run: in-sample sanity check on 40 train rows (head was fit on them)
            _E40 = embed(dict(spec, layers=sorted(float(v) for v in H['layers'])), train.text.values[:40])
            _acc = float(((((_E40 - H['mu']) / H['sd']) @ H['W'] + H['b']).argmax(1) == y[:40]).mean())
            log(f'  {spec["name"]} in-sample check acc {_acc:.2f}'); assert _acc > 0.5, f'{spec["name"]} head does not reproduce its fit'
        Z = (Ete - H['mu']) / H['sd']; zz = Z @ H['W'] + H['b']
        LP[spec['name']] = (zz - zz.max(1, keepdims=True) - np.log(np.exp(zz - zz.max(1, keepdims=True)).sum(1, keepdims=True))) * spec.get('w', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    if spec.get('type') == 'lora':
        log('lora', spec['name']); LP[spec['name']] = lora_predict(spec, test.text.values) * spec.get('w', 1.0)
        gc.collect(); torch.cuda.empty_cache(); log(spec['name'], 'done', gpu_mem()); continue
    name = spec['name']; cp = os.path.join(a.cache, f"{name}_{'sim' + a.sim + a.sim_k if a.sim else 'real'}.npy")
    if os.path.exists(cp): E = np.load(cp)
    else:
        log('embedding', name); E = embed(spec, all_text); np.save(cp, E)
    Etr, Ete = E[:len(train)], E[len(train):]
    r = torch_lr(Etr, y, Ete, spec.get('C', a.C))
    if a.perpassage:
        LP[name] = r * spec.get('w', 1.0)
        if a.adapt_w > 0:
            sizes = np.bincount(Gte)[Gte]; rows = sizes >= 2
            c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), spec.get('C', a.C))
            LP[name + '_twin'] = np.where(rows[:, None], c, 0.0) * a.adapt_w * spec.get('w', 1.0)
    else:
        c = torch_lr(center_groups(Etr, Gtr), y, center_groups(Ete, Gte), a.C)
        LP[name] = (r + c) * spec.get('w', 1.0)
    gc.collect(); torch.cuda.empty_cache(); log(name, 'done', gpu_mem())

# ---------------- 4. combine + assign
# typography guards: (mark, models that ever use it in train). Zero violations on the 38k matched pool for every rule.
GUARDS = [(' \u2014 ', {'claude-opus-4-6', 'claude-opus-5', 'glm-5.3', 'glm-5.3-flash'}),
          ('\u2019', {l for l in LABELS if not l.startswith(('claude', 'glm'))}),
          ('"', {'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'glm-5.3', 'glm-5.3-flash', 'gpt-5.5', 'kimi-k3'}),
          ('\u201c', {'gemini-3.8-flash', 'gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'kimi-k3', 'qwen3.8-max'}),
          ('\u2013', {'claude-opus-5', 'gemini-3.1-pro-preview', 'glm-5.3-flash', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'kimi-k3'})]
if not LP:raise RuntimeError('No completed main members')
for _key,_value in LP.items():finite(_key,_value)
tot=sum(LP.values())

COMP_P=None
if a.comp_w>0:
    if not a.comp_path or not os.path.exists(a.comp_path):raise RuntimeError('Required complementary output missing')
    _cm=json.load(open(a.comp_path.replace('comp_logit.npy','comp_meta.json')))
    if _cm['ids']!=test.id.astype(str).tolist():raise ValueError('Complement ID order differs')
    if sorted(_cm['labels'])!=sorted(LABELS) or len(_cm['labels'])!=18:raise ValueError('Complement labels differ')
    _cl=finite('complement',np.load(a.comp_path))[:,[_cm['labels'].index(l) for l in LABELS]]
    if _cl.shape!=(len(test),18):raise ValueError('Complement matrix shape differs')
    COMP_P=softmax(_cl,axis=1);log('complement loaded',_cm['seconds'])
wsum=sum(abs(v).max()>0 for v in LP.values())
core=tot/max(1,wsum) if a.sk_div else tot
scale=a.svc_w/max(1,wsum) if a.sk_div else a.svc_w
newcore=topic_correction(core,SVC_SCORES,Gte,a.topic_alpha,weight=scale)
P=softmax(core,axis=1);CAND=softmax(newcore,axis=1)
if COMP_P is not None:
    P=(1-a.comp_w)*P+a.comp_w*COMP_P;P/=P.sum(1,keepdims=True)
    CAND=(1-a.comp_w)*CAND+a.comp_w*COMP_P;CAND/=CAND.sum(1,keepdims=True)
out=Path(a.out).resolve();baseline=out.with_name('submission_baseline_0651.parquet')
status={'topic_alpha':a.topic_alpha,'active_main_members':list(LP),'pool_audit':POOL_AUDIT,'full_recipe_completed':False,
        'complement_loaded':COMP_P is not None,'atlas':[],'extra_gpu_forwards':0,
        'scope':'Baseline-recipe and candidate share ALL neural evidence. Neither has a new verified LB score.'}
def save_pair(phase):
    base_scores=final_scores(P,test.text.values,Gte,a.sinkhorn,bool(a.post),GUARDS)
    new_scores=final_scores(CAND,test.text.values,Gte,a.sinkhorn,bool(a.post),GUARDS)
    pb=base_scores.argmax(1);pn=new_scores.argmax(1)
    write_submission(test.id,[LABELS[k] for k in pb],baseline)
    write_submission(test.id,[LABELS[k] for k in pn],out)
    status.update(phase=phase,changed_rows=int((pb!=pn).sum()),rows=len(test),seconds=time.time()-T0)
    (out.parent/'run_report.json').write_text(json.dumps(status,indent=2))
    return new_scores
# A valid artifact exists before optional ATLAS starts; never blend half its rows.
save_pair('before_optional_atlas')
np.savez_compressed(out.parent/'main_evidence.npz',ids=test.id.to_numpy(dtype=str),labels=np.array(LABELS),
                    main_core=core,svc_scores=SVC_SCORES if SVC_SCORES is not None else np.empty((0,18)),groups=Gte,
                    pre_atlas=P,candidate_pre_atlas=CAND,svc_contribution=np.array(scale))
bundles=[d for d in a.atlas_dir.split(',') if d];weights=[float(w) for w in str(a.atlas_w).split(',')] if bundles else []
if len(weights)==1:weights*=len(bundles)
if len(weights)!=len(bundles) or sum(weights)>1:raise ValueError('Invalid ATLAS blend weights')
margin=np.sort(P,axis=1);margin=margin[:,-1]-margin[:,-2]
rows=np.where(margin<a.atlas_margin)[0] if a.atlas_margin>0 and len(test)>=50 else np.arange(len(test))
original=P.copy();original_candidate=CAND.copy()
for bi,(bundle,w) in enumerate(zip(bundles,weights)):
    record={'bundle':bundle,'weight':w,'selected_rows':len(rows)}
    if w<=0 or not len(rows):record['state']='not_needed';status['atlas'].append(record);continue
    deadline=min(float(os.environ.get('NTL_DEADLINE',T0+a.atlas_budget+120))-120,T0+a.atlas_budget)
    try:
        if a.sim or a.fake_test:raise RuntimeError('ATLAS is disabled for proxy validation and timing fixtures')
        if deadline-time.time()<30:raise TimeoutError('ATLAS reserve exhausted')
        request=out.parent/f'atlas_request_{bi}.npz';result=out.parent/f'atlas_result_{bi}.npz'
        result.unlink(missing_ok=True)
        np.savez_compressed(request,ids=test.id.to_numpy(dtype=str),rows=rows)
        cmd=[sys.executable,'-u',str(Path(__file__).with_name('atlas_worker.py')),'--data',a.data,'--bundle',bundle,
             '--base',a.atlas_base,'--request',str(request),'--out',str(result),'--deadline',str(deadline),
             '--bs',str(a.atlas_bs),'--chunk',str(a.atlas_chunk)]
        run_checked(cmd,deadline,out.parent/f'atlas_worker_{bi}.log')
        with np.load(result,allow_pickle=False) as ev:
            if not np.array_equal(ev['ids'],test.id.to_numpy(dtype=str)) or not np.array_equal(ev['rows'],rows):raise ValueError('ATLAS row mismatch')
            if ev['labels'].tolist()!=LABELS:raise ValueError('ATLAS label mismatch')
            A=finite('ATLAS evidence',ev['prob'])
            if A.shape!=(len(rows),18) or (A<0).any() or not np.allclose(A.sum(1),1,atol=1e-4):raise ValueError('Invalid ATLAS probabilities')
        P[rows]+=w*(A-original[rows]);CAND[rows]+=w*(A-original_candidate[rows])
        record['state']='mixed';log('ATLAS mixed at',w)
    except Exception as exc:
        record.update(state='skipped',error=repr(exc));log('ATLAS SKIPPED; keeping completed main + complement:',repr(exc))
    status['atlas'].append(record)
P/=P.sum(1,keepdims=True);CAND/=CAND.sum(1,keepdims=True)
status['full_recipe_completed']=(not a.sim and not a.fake_test and
    [s.get('name') for s in specs]==['q14v5','q8v5','ll8v5','n14b','q25_14','deb'] and
    COMP_P is not None and bool(bundles) and all(r['state'] in ('mixed','not_needed') for r in status['atlas']))
scores=save_pair('finished')
np.savez_compressed(out.parent/'final_evidence.npz',ids=test.id.to_numpy(dtype=str),labels=np.array(LABELS),baseline_prob=P,candidate_prob=CAND,final_scores=scores)
if a.save_lp:np.save(a.save_lp,log_softmax(scores,axis=1).astype(np.float32))
if a.sim:
    from sklearn.metrics import balanced_accuracy_score
    log('TRAIN-ONLY SVC PROXY balanced accuracy',balanced_accuracy_score(test_truth.model.map(L2I),scores.argmax(1)))
log('wrote',str(out),status)


In [ ]:
%%writefile "{RUNTIME_DIR}/comp_stack.py"
"""Complementary frozen-feature stack (public Apache-2.0 notebooks by prvsiyan / lucasboesen / vnton37nguyn / kirilltym,
as embedded in the public 0.6xx ATLAS notebooks). Run as a separate process; writes its raw log-probability blend."""
# ============================================================ 0. configuration
import gc, glob, json, os, pathlib, re, time
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',TOKENIZERS_PARALLELISM='false')
from ntl_upgrade import finite,clean_pool
COMPLETED_MEMBERS=[]
import numpy as np, pandas as pd
try:
    import torch
except ImportError:   # local smoke without torch
    torch = None
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.sparse import csr_matrix, diags, hstack, vstack
from scipy.spatial.distance import squareform
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

SMOKE = os.environ.get('NTL_SMOKE') == '1'   # local code-path check: pseudo features instead of checkpoints, CPU
T0 = time.time()
LABELS = sorted(['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra',
                 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash',
                 'claude-fable-5', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5',
                 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3'])
K = len(LABELS)
LAB2I = {l: i for i, l in enumerate(LABELS)}
FAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in ('claude', 'gpt', 'gemini', 'glm')}

# ---- test-pool grouping
GROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage
STYLE_GROUP_THRESH = 0.90                                           # stylegram's own median-normalised content distance

# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)
W = {'stylegram': 1.516, 'stylometry_lgb': 0.063, 'ultrabert': 0.095,
     'q3_8b': 0.0, 'q25_7b': 0.0, 'q3_4b': 0.0, 'glm9b': 0.0, 'debv3l': 0.05, 'electl': 0.032, 'desk': 0.132,
     'lora24_public': 0.177, 'lora24_seed7': 0.113, 'lora24_claude': 0.177, 'lora20_q25_instyle': 0.17,
     'tokattn_q35': 0.153, 'tokattn_deb': 0.185}
TWIN_W = {'q3_8b': 1.5 * 0.219, 'q25_7b': 1.5 * 0.141, 'q3_4b': 1.5 * 0.083, 'glm9b': 1.5 * 0.110, 'debv3l': 0.25}
LORA_TWIN_W = {'lora24_public': 0.15, 'lora24_claude': 0.15}
GATE_LAM = {'claude_style': 1.5, 'claude_lora': 1.0, 'family': 0.25}
BALANCE_ALPHA = 0.5
EXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5

# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,
#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)
GAUSS_HEADS = {'gauss_q3_8b_L22': dict(src='q3_8b', part='L22_mean', d=384, T1=2.351, Tg=2.306, w=TWIN_W['q3_8b']),
               'gauss_glm9b_L8': dict(src='glm9b', part='L8_mean', d=384, T1=2.722, Tg=2.596, w=TWIN_W['glm9b']),
               'gauss_q3_4b_L4': dict(src='q3_4b', part='L4_mean', d=384, T1=2.725, Tg=2.578, w=TWIN_W['q3_4b']),
               'gauss_debv3l_L24': dict(src='debv3l', part='L24_mean', d=384, T1=3.112, Tg=3.069, w=TWIN_W['debv3l'])}
GAUSS_SETTING = dict(red='wpca', se_shrink='lw', sp_shrink='lw', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)

# ---- member specifications
LM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C
    ('glm9b', 'glm4', 4096, [(8, 'mean'), (3, 'mean')], [(GROUP_LAYER, 'mean')], 0.0025),   # first: its L24 builds the groups
    ('q3_8b', 'qwen3', 4096, [(7, 'mean'), (22, 'mean')], [], 0.0025),
    ('q25_7b', 'qwen2', 3584, [(6, 'mean'), (20, 'mean')], [], 0.0025),
    ('q3_4b', 'qwen3', 2560, [(4, 'style'), (7, 'style')], [(4, 'mean')], 0.0025)]
ENC_HEADS = [  # key, input directory name, parts, C
    ('debv3l', 'deberta-v3-large', [(24, 'mean'), (6, 'mean')], 0.005),
    ('electl', 'electra-large-discriminator', [(21, 'mean'), (24, 'style')], 0.005),
    ('desk', 'desklib-detector', [(23, 'mean'), (23, 'style')], 0.005)]
LORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]
    (('qwen3', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [
        ('lora24_public', '**/lora24/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz'),
        ('lora24_seed7', '**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz'),
        ('lora24_claude', '**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json', 'heads.safetensors', 'anchor_features.npz')]),
    (('qwen2', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [
        ('lora20_q25_instyle', '**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json', 'norms.safetensors', 'valid_features.npz')])]
LORA_MAX_LEN, LORA_BS = 447, 8
TOKATTN_HEADS = [('tokattn_q35', 'tokattn_final.pt', 'qwen3_5'), ('tokattn_deb', 'tokattn_debl23_final.pt', 'deberta')]
if os.environ.get('NTL_DROP_Q35') == '1':   # Qwen3.5-9B token-attention member: 1.7-2.7 h fixed cost on Kaggle, weight 0.153 of ~3.3
    TOKATTN_HEADS = [t for t in TOKATTN_HEADS if t[0] != 'tokattn_q35']

# ---- data
if SMOKE:
    _all = pd.read_csv(os.environ['NTL_DATA'] + '/train.csv')
    _p = _all['prompt_id'].drop_duplicates().sample(frac=1, random_state=0).tolist()
    train = _all[_all['prompt_id'].isin(_p[:30])].reset_index(drop=True)
    test = _all[_all['prompt_id'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)
    test = pd.concat([test[test['prompt_id'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)
    if os.environ.get('NTL_TINY') == '1':
        test = test.head(3).reset_index(drop=True)
    SMOKE_Y = test['model'].map(LAB2I).to_numpy(dtype=int)
else:
    DATA_DIR = next(p for p in [pathlib.Path('/kaggle/input/competitions/name-that-llm'), pathlib.Path('/kaggle/input/name-that-llm')]
                    if (p / 'train.csv').exists())
    train, test = pd.read_csv(DATA_DIR / 'train.csv'), pd.read_csv(DATA_DIR / 'test.csv')
    if os.environ.get('NTL_FAKE_TEST'):   # timing probe: pool passages as a stand-in test (same rows/seed as infer.py --fake_test)
        test = pd.read_parquet(os.environ['NTL_POOL']).sample(int(os.environ['NTL_FAKE_TEST']), random_state=0)[['text']].reset_index(drop=True)
        test['id'] = [f'fake{i}' for i in range(len(test))]
test['text'] = test['text'].fillna('').astype(str)
TR, TE = train['text'].tolist(), test['text'].tolist()
y = train['model'].map(LAB2I).to_numpy(dtype=int)
PROMPT = train['prompt_id'].to_numpy(dtype=object)
TINY = os.environ.get('NTL_PARITY_CHECK')=='1' and len(TE)<50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code
DEV = 'cuda' if torch is not None and torch.cuda.is_available() else 'cpu'
print(f'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}', flush=True)

LOGIT = np.zeros((len(TE), K))           # the running log-probability blend


def add_member(key, prob, rows=None, weight=None):
    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""
    w = W[key] if weight is None else weight
    finite(key,prob)
    COMPLETED_MEMBERS.append(key)
    lp = np.log(np.clip(prob, 1e-9, 1.0))
    if rows is None:
        LOGIT[:] += w * lp
    else:
        LOGIT[rows] += w * lp
    print(f'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)', flush=True)


def softmax_rows(S):
    S = S - S.max(1, keepdims=True)
    P = np.exp(S)
    return P / P.sum(1, keepdims=True)


def full_prob(P_part, classes):
    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""
    P = np.zeros((len(P_part), K))
    P[:, classes] = P_part
    return P


def group_centre_dense(X, groups):
    X = np.asarray(X, dtype=np.float32).copy()
    for g in np.unique(groups):
        i = groups == g
        X[i] -= X[i].mean(0)
    return X


def self_test_rows():
    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""
    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]
    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]

# ============================================================ 1. grouping the test pool
def average_linkage_groups(D, thresh):
    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""
    if len(D) < 2:
        return np.ones(len(D), dtype=int)
    D = np.asarray(D, dtype=np.float64).copy()
    np.fill_diagonal(D, 0.0)
    return fcluster(linkage(squareform(D, checks=False), method='average'), thresh, criterion='distance')


def content_cosine_distance(texts, max_features=None):
    X = TfidfVectorizer(stop_words='english', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)
    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)


def mixed_groups(texts, E, thresh=GROUP_THRESH):
    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    Z = np.asarray(E, dtype=np.float64)
    Z = Z - Z.mean(0)
    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12
    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))
    return average_linkage_groups(D, thresh)


def normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):
    """The stylegram's grouping: content distance divided by its median off-diagonal value, cut at thresh."""
    if len(texts) < 2:
        return np.ones(len(texts), dtype=int)
    D = content_cosine_distance(texts, max_features=100_000)
    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)
    return average_linkage_groups(D / med, thresh)


def eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):
    size = np.bincount(groups)[groups]
    return (size >= lo) & (size <= hi)


def describe_groups(name, groups):
    size = np.bincount(groups)[groups]
    print(f'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}'
          f'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}', flush=True)


G_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)

# ============================================================ 2. style n-grams + extra passages + gates
FUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "
                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "
                           "without for of by over under between through into about is are was were be been being have has had do does did "
                           "can could will would shall should may might must not no nor all any some each every both either neither other "
                           "another such only also just very too more most less least here there when where why how what whether then now "
                           "still yet even much many few little".split())
TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def shape_view(text):
    return re.sub(r"[^\W\d_]+", lambda m: 'X' if m[0].isupper() else 'x', text)


def function_view(text):
    return ' '.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else 'CONTENT' for t in TOKEN_RE.findall(text))


class StyleGrams:
    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""
    VIEWS = [('char', str, (3, 5), 250_000, 1.0), ('word', str, (1, 3), 180_000, 0.7),
             ('char', shape_view, (2, 6), 60_000, 0.5), ('word', function_view, (1, 3), 50_000, 0.5)]

    def fit(self, texts):
        self.vecs = []
        for analyzer, view, ngr, maxf, w in self.VIEWS:
            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)
            if analyzer == 'word':
                opts['token_pattern'] = r"(?u)\b\w+\b|[^\w\s]"
            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))
        return self

    def transform(self, texts):
        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format='csr')


def sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):
    """Subtract each group's mean row (sparse); groups outside [lo, hi] are left as they are."""
    X = csr_matrix(X)
    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)
    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))
    if not len(keep):
        return X.copy()
    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))
    rows = np.flatnonzero(col[inv] >= 0)
    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))
    means = diags(1.0 / cnt[keep]) @ (M.T @ X)
    return (X - M @ means).tocsr()


def svc_probs(svc, X):
    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)


# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels
def load_extra_passages():
    sources = [('**/ntl_extra_passages_claude.csv', 'model', None), ('**/gen_passages.csv', 'model', None),
               ('**/ntl_generations.csv', 'label', 'finish_reason'), ('**/ntl_proxy_generations.csv', 'label', None),
               ('**/ntl_generated*.csv', 'model', 'finish'), ('**/passages_clean.csv', 'model', None)]
    parts = []
    for pattern, label_col, finish_col in sources:
        for f in sorted(glob.glob('/kaggle/input/' + pattern, recursive=True)):
            d = pd.read_csv(f)
            if finish_col and finish_col in d:
                d = d[d[finish_col].eq('stop')]
            d = d.rename(columns={label_col: 'model'})[['prompt_id', 'model', 'text']].copy()
            d['prompt_id'] = ('pv_' if f.endswith('passages_clean.csv') else '') + d['prompt_id'].astype(str)   # prvsiyan's prompts are a different bank
            parts.append(d)
    if not parts:
        return pd.DataFrame(columns=['prompt_id', 'model', 'text'])
    x = pd.concat(parts, ignore_index=True)
    x = x[x['model'].isin(LABELS)].copy()
    x['text'] = x['text'].astype(str).map(lambda t: re.sub(r'\s+', ' ', re.sub(r'^\*\*(.*)\*\*$', r'\1', t.strip())).strip())
    x = x[x['text'].str.split().str.len().between(30, 300)].drop_duplicates(subset=['model', 'text'])
    x,overlap_audit=clean_pool(x,TE)
    print('Complement external-overlap audit:',overlap_audit,flush=True)
    x = x.drop_duplicates(subset=['prompt_id', 'model'])
    x = x[x.groupby('prompt_id')['model'].transform('nunique') >= EXTRA_MIN_LABELS].reset_index(drop=True)
    return x


EXTRA = load_extra_passages()
EX_T, EX_Y, EX_G = EXTRA['text'].tolist(), EXTRA['model'].map(LAB2I).to_numpy(dtype=int), EXTRA['prompt_id'].to_numpy(dtype=object)
print(f'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts', flush=True)


def extra_sample_weights():
    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class's extra rows share EXTRA_SHARE of it."""
    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)
    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)
    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)
    return np.concatenate([w_tr, w_ex])


grams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra
X_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)


def fit_svc(X, labels, C=0.5, **kw):
    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)


if X_ex is not None:
    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())
else:
    svc_plain = fit_svc(X_tr, y)
P_style = svc_probs(svc_plain, X_te)

# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups
X_tr_c = sparse_group_centre(X_tr, PROMPT)
if X_ex is not None:
    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),
                          sample_weight=extra_sample_weights())
else:
    svc_centred = fit_svc(X_tr_c, y)
G_style = normalised_content_groups(TE)
rows_style = eligible(G_style)
describe_groups('stylegram groups', G_style)
if rows_style.any():
    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))
if TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)
    st_idx, st_g = self_test_rows()
    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])
    print(f'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}', flush=True)
add_member('stylegram', P_style)
del X_tr_c, svc_plain, svc_centred, P_style


# ---- gates: style SVCs that only re-rank inside one family after the blend
def family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):
    g = StyleGrams().fit(texts)
    svc = LinearSVC(C=C, class_weight='balanced' if balanced else None, random_state=42, max_iter=4000).fit(
        g.transform(texts), labels, sample_weight=sample_weight)
    assert list(svc.classes_) == list(classes)
    d = svc.decision_function(g.transform(TE))
    return np.column_stack([-d, d]) if len(classes) == 2 else d


GATES = {}  # The scored program never consumed these family-gate scores.
del grams, X_tr, X_ex, X_te

# ============================================================ 3. stylometric features + LightGBM
PUNCT = list(',.;:!?()[]"\'-') + ['—', '–', '‘', '’', '“', '”', '…', '*', '/', '%']
STOPS = sorted(ENGLISH_STOP_WORDS)
SENT_RE, WORD_RE = re.compile(r'(?<=[.!?])\s+'), re.compile(r"[A-Za-z’']+")


def stylometry(t):
    words = WORD_RE.findall(t)
    low = [w.lower() for w in words]
    nw, nc = max(len(words), 1), max(len(t), 1)
    sents = [s for s in SENT_RE.split(t.strip()) if s]
    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)
    wl = np.array([len(w) for w in words] or [0], dtype=float)
    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,
         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,
         t.count(', ') / nw, t.count(' and ') / nw, t.count(' but ') / nw,
         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in ('this', 'these', 'it', 'they')) / max(len(sents), 1),
         sum(w.endswith('ly') for w in words) / nw, sum(w.endswith('ing') for w in words) / nw, sum(w.endswith('tion') for w in words) / nw]
    f += [t.count(q) / nw for q in PUNCT]
    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)
    f += [counts.get(s, 0) / nw for s in STOPS]
    return f


F_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))
try:
    import lightgbm as lgb
    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)
    add_member('stylometry_lgb', full_prob(booster.predict_proba(F_te), booster.classes_))
    del booster
except ImportError:
    assert SMOKE, 'lightgbm missing'
    print('smoke: lightgbm unavailable, stylometry member skipped', flush=True)
del F_tr, F_te

# ============================================================ 4b. shared-prompt Gaussian group head
# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.
# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):
#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).
# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint
# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over
# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected
# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).
import itertools
K = 18
DEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,
           "iters": 5, "repel": True, "sp_scale": 1.0}

# =================================================================== primitives ====
def _lse(A, axis=None, keepdims=False):
    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m
    if keepdims: return out
    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())

def log_softmax(L): return L - _lse(L, axis=1, keepdims=True)

def _lw_shrinkage(X):
    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""
    from sklearn.covariance import ledoit_wolf_shrinkage
    return float(ledoit_wolf_shrinkage(X, assume_centered=True))

def _shrunk(S, a):
    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)

def _two_way(Z, y, pinv, G, iters=4):
    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""
    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)
    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))
    for _ in range(iters):
        R = Z - P[pinv]
        for c in cls: mu[c] = R[y == c].mean(0)
        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]
    return mu, P, cnt_p

def fit(X, y, prompts, cfg=None):
    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.
    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller's cross-fit)."""
    c = dict(DEFAULT); c.update(cfg or {})
    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape
    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)
    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx
    if c["red"] == "none":
        V = np.eye(D)
    else:
        if c["red"] == "wpca":
            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]
        elif c["red"] == "pca": R = Xs
        else: raise ValueError(c["red"])
        d = int(min(c["d"], N - G - 1, D))
        # economy SVD through the (N x N) Gram matrix (N << D for the real views)
        if N < D:
            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]
            V = (R.T @ U) / np.sqrt(w)
        else:
            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T
    Z = Xs @ V; d = Z.shape[1]
    mu, P, ng = _two_way(Z, y, pinv, G)
    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)
    Se_raw = E.T @ E / dof
    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)
    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw
    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])
    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]
    # joint diagonalisation: A Se A' = I, A Sp A' = diag(lam)
    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)
    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li
    W = V @ A.T
    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,
            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}

def transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]

_MASK = {}
def _inj_mask(n, C):
    if (n, C) not in _MASK:
        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)
        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]
        _MASK[(n, C)] = bad
    return _MASK[(n, C)]

def singleton_ll(U, m, lam):
    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""
    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)

def joint_terms(U, m, lam):
    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""
    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]
    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)
    return a, B

def joint_table(a, B):
    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""
    n, C = a.shape; Q = np.zeros([C] * n)
    for i in range(n):
        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)
    for i, j in itertools.combinations(range(n), 2):
        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)
    return Q

def exact_marginals(Q, T=1.0, inj=True):
    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""
    n = Q.ndim; C = Q.shape[0]; Qt = Q / T
    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)
    lz = _lse(Qt); out = np.empty((n, C))
    for i in range(n):
        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz
    return out

def em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):
    """cavity EM for one group (any n >= 2): returns (n, C) log q."""
    n = len(U); v = lam / (1 + (n - 1) * lam)
    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))
    q = np.exp(log_softmax(L))
    for _ in range(iters):
        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)
        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T
        if repel:   # mean-field at-most-once: discourage classes the other members already hold
            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)
        q = np.exp(log_softmax(L))
    return log_softmax(L)

LAM_DIFF = 1e6
def lam_g(prm, cfg=None):
    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg 'diff'): then the joint likelihood
    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs
    centred prototypes) and the prompt term cancels exactly."""
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})
    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]

def group_logpost(U, prm, T1=None, Tg=None, cfg=None):
    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)
    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg
    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)
    if n <= c["n_exact"]:
        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])
    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])

def scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):
    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""
    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))
    sizes = np.bincount(lab); s1 = sizes[lab] == 1
    T1 = prm["T1"] if T1 is None else T1
    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)
    for g in np.where(sizes >= 2)[0]:
        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)
    return out

# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads
if not SMOKE:
    from transformers import AutoConfig, AutoModel, AutoTokenizer
CACHED_PARTS = {}   # key -> {part name: (train features, test features)}


def find_checkpoint(model_type, hidden=None, dir_name=None):
    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""
    pattern = f'/kaggle/input/**/{dir_name}/config.json' if dir_name else '/kaggle/input/**/config.json'
    for cfg in sorted(glob.glob(pattern, recursive=True)):
        try:
            c = json.load(open(cfg))
        except Exception:
            continue
        if dir_name or (c.get('model_type') == model_type and (hidden is None or c.get('hidden_size') == hidden)):
            return os.path.dirname(cfg)
    return None


def load_truncated_decoder(path, keep, dtype):
    cfg = AutoConfig.from_pretrained(path)
    cfg.num_hidden_layers = keep
    if getattr(cfg, 'layer_types', None):
        cfg.layer_types = cfg.layer_types[:keep]
    kw = dict(config=cfg, dtype=dtype)
    if DEV == 'cuda':
        kw['device_map'] = 'auto'
    return AutoModel.from_pretrained(path, **kw).eval()


def style_token_mask(tok):
    """Vocabulary mask of 'style' tokens: the decoded piece is an English stop word or pure punctuation."""
    pieces = [tok.decode([i]) for i in range(len(tok))]
    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r'\s*[^\w\s]+\s*', p)) for p in pieces])


@torch.no_grad()
def pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):
    """{'L{layer}_{pool}': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""
    H = model.config.hidden_size
    out = {f'L{L}_{pool}': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}
    smask = style_token_mask(tok) if any(pool == 'style' for _, pool in parts) else None
    dev = next(model.parameters()).device
    order = np.argsort([len(t) for t in texts])
    retried = 0
    for b0 in range(0, len(texts), bs):
        idx = order[b0:b0 + bs]
        if decoder:
            batch = [(tok.bos_token or '') + texts[i] for i in idx]
            enc = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)
        else:
            enc = tok([texts[i] for i in idx], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(dev)
        hs = model(**enc, output_hidden_states=True).hidden_states
        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16
            retried += 1
            model.to(torch.bfloat16)
            hs = model(**enc, output_hidden_states=True).hidden_states
        m_all = enc['attention_mask'].float()
        for L, pool in parts:
            m = m_all * smask.to(m_all.device)[enc['input_ids']].float() if pool == 'style' else m_all
            m = m.unsqueeze(-1).to(hs[L].device)
            out[f'L{L}_{pool}'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
        del hs, enc
    if retried:
        print(f'  {retried} batches redone in bf16', flush=True)
    return {k: np.nan_to_num(v) for k, v in out.items()}


def pseudo_parts(parts, dim=48):
    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""
    from sklearn.decomposition import TruncatedSVD
    tf = TfidfVectorizer(analyzer='char', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)
    out = {}
    for j, (L, pool) in enumerate(parts):
        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))
        out[f'L{L}_{pool}'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))
    return out


def extract_parts(key, parts, loader):
    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""
    if SMOKE:
        return pseudo_parts(parts)
    model, tok, decoder, bs = loader()
    tr, te = pooled_states(model, tok, TR, parts, decoder, bs), pooled_states(model, tok, TE, parts, decoder, bs)
    del model, tok
    gc.collect()
    if DEV == 'cuda':
        torch.cuda.empty_cache()
    return {k: (tr[k], te[k]) for k in tr}


def decoder_loader(model_type, hidden, keep):
    def load():
        path = find_checkpoint(model_type, hidden)
        assert path, f'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input'
        tok = AutoTokenizer.from_pretrained(path)
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        return load_truncated_decoder(path, keep, torch.float16 if DEV == 'cuda' else torch.float32), tok, True, 8
    return load


def encoder_loader(dir_name):
    def load():
        path = find_checkpoint(None, dir_name=dir_name)
        assert path, f'encoder directory {dir_name} not found under /kaggle/input'
        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16
    return load


def concat_parts(cache, parts):
    return (np.concatenate([cache[f'L{L}_{p}'][0] for L, p in parts], 1), np.concatenate([cache[f'L{L}_{p}'][1] for L, p in parts], 1))


def plain_and_twin(key, Etr, Ete, C):
    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""
    if W[key] > 0:
        sc = StandardScaler().fit(Etr)
        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)
        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)
        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))
    else:
        print(f'  {key}: plain head has weight 0 in the 0.563 blend, not fitted', flush=True)
    if key in TWIN_W:
        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))
        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)
        if GROUP_ROWS.any():
            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))
            add_member(key + '_twin', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])
        if TINY:
            st_idx, st_g = self_test_rows()
            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))
            print(f'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}', flush=True)


def gaussian_group_head(name, spec, Xtr, Xte):
    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""
    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec['d']))
    prm['T1'], prm['Tg'] = spec['T1'], spec['Tg']
    print(f'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}', flush=True)
    if GROUP_ROWS.any():
        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)
        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec['w'])
    if TINY:
        st_idx, st_g = self_test_rows()
        LP = scores(Xtr[st_idx], st_g, prm)
        print(f'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}', flush=True)


for key, model_type, hidden, parts, extra_parts, C in LM_HEADS:
    print(f'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}', flush=True)
    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))
    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups
        G_TE = mixed_groups(TE, cache[f'L{GROUP_LAYER}_mean'][1])
        GROUP_ROWS = eligible(G_TE)
        describe_groups('test groups (content + GLM L24)', G_TE)
        if SMOKE:
            same_g, same_p = G_TE[:, None] == G_TE[None, :], test['prompt_id'].to_numpy(dtype=object)[:, None] == test['prompt_id'].to_numpy(dtype=object)[None, :]
            off = ~np.eye(len(TE), dtype=bool)
            print(f'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} '
                  f'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}', flush=True)
    CACHED_PARTS[key] = cache
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    CACHED_PARTS.pop(key)
    gc.collect()

for key, dir_name, parts, C in ENC_HEADS:
    print(f'[{key}] {dir_name} parts {parts}', flush=True)
    cache = extract_parts(key, parts, encoder_loader(dir_name))
    plain_and_twin(key, *concat_parts(cache, parts), C)
    for name, spec in GAUSS_HEADS.items():
        if spec['src'] == key:
            gaussian_group_head(name, spec, *cache[spec['part']])
    del cache
    gc.collect()

# ============================================================ 5. LoRA classifiers (inference only)
@torch.no_grad()
def score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):
    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;
    within-Claude readout -> Claude gate."""
    import importlib.util as iu
    _find = iu.find_spec   # Kaggle's torchao build is rejected by peft; hide it (unused here)
    iu.find_spec = lambda name, *a, **k: None if name.split('.')[0] == 'torchao' else _find(name, *a, **k)
    from peft import PeftModel
    import peft.import_utils as pi
    pi.is_torchao_available = lambda *a, **k: False
    from safetensors.torch import load_file
    model_type, H, keep = trunk_spec
    base = find_checkpoint(model_type, H)
    assert base, f'no {model_type} base checkpoint under /kaggle/input'
    cfg = AutoConfig.from_pretrained(base)
    cfg.num_hidden_layers, cfg.use_cache = keep, False
    if getattr(cfg, 'layer_types', None) is not None:
        cfg.layer_types = cfg.layer_types[:keep]
    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map='auto').eval()
    for key, marker, norms_file, anchor_file in checkpoints:
        hits = sorted(glob.glob('/kaggle/input/' + marker, recursive=True))
        if not hits:   # upstream training kernel no longer public: drop this member, keep the rest of the stack
            if key!='lora24_public':raise FileNotFoundError(f'Required complementary member missing: {key}')
            print(f'  SKIP {key}: historical parent checkpoint unavailable ({marker})',flush=True);continue
        ck = pathlib.Path(os.path.dirname(hits[0]))
        print(f'[{key}] {ck}', flush=True)
        tok = AutoTokenizer.from_pretrained(ck / 'tokenizer')
        tok.padding_side = 'right'
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        lm = PeftModel.from_pretrained(trunk, ck / 'adapter').eval()
        state = load_file(str(ck / norms_file))
        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])
        norms.load_state_dict({k[len('norms.'):]: v for k, v in state.items() if k.startswith('norms.')})
        ro = np.load(ck / 'readouts.npz')
        assert list(ro['classes']) == LABELS and list(ro['pool_layers']) == pool_layers
        dev = lm.get_input_embeddings().weight.device
        norms = norms.to(dev).float()

        def token_ids(texts):
            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)['input_ids']]   # these checkpoints' anchors match the tokenizer's own eos

        def features(texts, bs):
            ids = token_ids(texts)
            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)
            order = np.argsort([len(t) for t in ids], kind='stable')
            for b0 in range(0, len(order), bs):
                idx = order[b0:b0 + bs]
                batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
                with torch.autocast('cuda', dtype=torch.float16):
                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states
                m = batch['attention_mask'].unsqueeze(-1).float()
                m[:, 0] = 0   # the first token is masked out of the mean, as in training
                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))
                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()
                del hs, batch
            return Z

        if (ck / anchor_file).exists():   # the checkpoint's own features for training rows 0..2: our forward pass must reproduce them
            saved = np.load(ck / anchor_file)['features'].astype(np.float32)
            ours = features(TR[:3], 3)
            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))
            print(f'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}', flush=True)
            assert rel < 0.02, f'{key}: our features do not match the checkpoint'
        Z = features(TE, LORA_BS)
        add_member(key, softmax_rows(Z @ ro['plain_coef'].T + ro['plain_intercept']))
        if 'claude_coef' in ro.files:
            lc = Z @ ro['claude_coef'].T + ro['claude_intercept']
            GATES['claude_lora'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)
        if key in LORA_TWIN_W and GROUP_ROWS.any():
            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])
            add_member(key + '_twin', softmax_rows(Zc @ ro['center_coef'].T + ro['center_intercept']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])
        trunk = lm.unload()
        del lm, Z
        gc.collect()
        torch.cuda.empty_cache()
    del trunk
    gc.collect()
    torch.cuda.empty_cache()


if not SMOKE:
    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:
        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)

# ============================================================ 6. token-attention heads + UltraBERT
class TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)
    def __init__(self, k, h=4096):
        super().__init__()
        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)

    def forward(self, x, m):
        h = torch.tanh(self.proj(self.drop(x)))
        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)
        mf = m.unsqueeze(-1).float()
        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))


@torch.no_grad()
def token_attention_probs(texts, head_file, backbone, bs=8):
    hits = sorted(glob.glob('/kaggle/input/**/' + head_file, recursive=True))
    assert hits, f'training-kernel output holding {head_file} not attached'
    d = torch.load(hits[0], map_location='cpu')
    assert list(d['classes']) == LABELS
    decoder = d.get('decoder', True)
    path = find_checkpoint('qwen3_5') if backbone == 'qwen3_5' else find_checkpoint(None, dir_name='deberta-v3-large')
    assert path, f'backbone {backbone} not attached'
    cfg = AutoConfig.from_pretrained(path)
    if decoder:
        tc = getattr(cfg, 'text_config', None) or cfg
        tc.num_hidden_layers = d['keep']
        if getattr(tc, 'layer_types', None):
            tc.layer_types = tc.layer_types[:d['keep']]
        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map='auto').eval()
    else:
        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()
    tok = AutoTokenizer.from_pretrained(path)
    tok.padding_side = 'right'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    H = d.get('hidden', 4096)
    net = TokenAttention(K, H)
    net.load_state_dict(d['state_dict'])
    dev = lm.get_input_embeddings().weight.device
    net, mu, sd = net.to(dev).eval(), d['mu'].float().to(dev), d['sd'].float().to(dev)
    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d['max_len'])['input_ids']
    P = np.zeros((len(texts), K))
    order = np.argsort([len(t) for t in ids])
    for b0 in range(0, len(order), bs):
        idx = order[b0:b0 + bs]
        batch = tok.pad({'input_ids': [ids[i] for i in idx]}, return_tensors='pt').to(dev)
        out = lm(input_ids=batch['input_ids'], attention_mask=batch['attention_mask'], output_hidden_states=True)
        hs = (out.hidden_states if getattr(out, 'hidden_states', None) is not None else out.language_model_outputs.hidden_states)[d['layer']]
        n = batch['attention_mask'].sum(1)
        T = max(int(n.max()) - 1, 1)
        x = torch.zeros(len(idx), T, H, device=dev)
        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)
        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training
            k = int(n[r]) - 1
            if k > 0:
                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)
                m[r, :k] = True
        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()
        del hs, out, batch
    del lm
    gc.collect()
    torch.cuda.empty_cache()
    return P


@torch.no_grad()
def ultrabert_probs(texts):
    from transformers import AutoModelForSequenceClassification
    folds = None
    for root in glob.glob('/kaggle/input/**/ultrabert-380m-ntl', recursive=True):
        fs = [f for f in sorted(glob.glob(os.path.join(root, 'fold*'))) if os.path.exists(os.path.join(f, 'config.json'))
              and json.load(open(os.path.join(f, 'config.json'))).get('model_type') == 'ultrabert']
        if len(fs) == 5:
            folds = fs
            break
    assert folds, 'ultrabert-380m-ntl (5 folds) not attached'
    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)
    ids = tok(texts, truncation=True, max_length=512)['input_ids']
    order = np.argsort([len(e) for e in ids])
    P = np.zeros((len(texts), K))
    for fp in folds:
        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation='sdpa').to(DEV).eval()
        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]
        for b0 in range(0, len(order), 32):
            idx = order[b0:b0 + 32]
            b = tok.pad({'input_ids': [ids[j] for j in idx]}, return_tensors='pt').to(DEV)
            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)
        del em
        gc.collect()
        torch.cuda.empty_cache()
    return P


if not SMOKE:
    for key, head_file, backbone in TOKATTN_HEADS:
        if TINY:   # in-sample check on training rows the head was fitted on: must be high, or the checkpoint is not being reproduced
            acc = float(np.mean(token_attention_probs(TR[:40], head_file, backbone).argmax(1) == y[:40]))
            print(f'  {key} in-sample check on 40 training rows: acc {acc:.2f}', flush=True)
            assert acc > 0.5, f'{key} does not reproduce its training fit'
        add_member(key, token_attention_probs(TE, head_file, backbone))
    add_member('ultrabert', ultrabert_probs(TE))



finite('complete complementary stack',LOGIT)
np.save('/kaggle/working/comp_logit.npy', LOGIT)
json.dump({'labels': LABELS, 'ids': test['id'].astype(str).tolist(), 'seconds': time.time() - T0,'completed_members':COMPLETED_MEMBERS}, open('/kaggle/working/comp_meta.json', 'w'))
print(f'COMP STACK DONE {time.time() - T0:.0f}s', flush=True)


In [ ]:
%%writefile "{RUNTIME_DIR}/atlas_worker.py"
"""The parent's ATLAS scoring formula in an isolated, time-limited process."""
import argparse,json,os,sys,time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from ntl_upgrade import LABELS,finite

def main():
    p=argparse.ArgumentParser()
    for key in ['data','bundle','base','request','out']:p.add_argument('--'+key,required=True)
    p.add_argument('--deadline',type=float,required=True);p.add_argument('--bs',type=int,default=4);p.add_argument('--chunk',type=int,default=3)
    a=p.parse_args();req=np.load(a.request,allow_pickle=False);rows=req['rows']
    test=pd.read_csv(Path(a.data)/'test.csv',dtype={'id':str});test['text']=test.text.fillna('')
    assert np.array_equal(test.id.to_numpy(dtype=str),req['ids'])
    sys.path.insert(0,a.bundle)
    from ntl_atlas.model import load_model
    from ntl_rlcd.data import LABELS as alabels
    from safetensors.torch import load_file
    cfg=json.load(open(Path(a.bundle)/'atlas_bundle.json'))['training_config']
    model,tok=load_model(a.base,alabels,cfg,device='auto')
    model.load_trainable(load_file(str(Path(a.bundle)/'adapters.safetensors')));model.eval()
    enc=tok(test.text.values[rows].tolist(),truncation=True,max_length=cfg['max_length'])['input_ids']
    order=np.argsort([len(t) for t in enc],kind='stable');out=np.zeros((len(rows),18),np.float32)
    ctok=np.cumsum(np.array([max(1,len(enc[int(i)])) for i in order],np.float64));start=time.time()
    for b in range(0,len(order),a.bs):
        if time.time()>=a.deadline:raise TimeoutError('ATLAS deadline before batch')
        ix=order[b:b+a.bs];pack=tok.pad({'input_ids':[enc[int(i)] for i in ix]},padding=True,return_tensors='pt').to(model.device)
        with torch.inference_mode():prob,_=model.score(pack['input_ids'],pack['attention_mask'],source_chunk=a.chunk)
        out[ix]=prob.float().cpu().numpy();done=b+len(ix)
        if b%(25*a.bs)==0:
            projected=time.time()+(time.time()-start)/ctok[done-1]*(ctok[-1]-ctok[done-1])*1.1
            print(f'ATLAS {done}/{len(rows)} projected seconds remaining {projected-time.time():.0f}',flush=True)
            if done>50 and projected>a.deadline:raise TimeoutError('ATLAS projection exceeds save reserve')
    finite('ATLAS',out)
    if not np.allclose(out.sum(1),1,atol=1e-4):raise ValueError('ATLAS probabilities do not sum to one')
    np.savez_compressed(a.out,rows=rows,ids=req['ids'],labels=np.array(LABELS),prob=out[:,[alabels.index(l) for l in LABELS]])
if __name__=='__main__':main()


In [ ]:
%%writefile "{RUNTIME_DIR}/validate_upgrade.py"
"""Nested, prompt-held-out validation of the exact main SVC only.

No released task-trained adapters or public pool is honest OOF for train.csv.
Consequently these numbers measure a train-only SVC proxy, not the full stack.
"""
from pathlib import Path
import argparse, json, hashlib, sys, time
import numpy as np
import pandas as pd
from scipy.special import log_softmax
from sklearn.svm import LinearSVC
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score
sys.path.insert(0,str(Path(__file__).resolve().parent/'enhanced_source'))
from ntl_upgrade import FourViews,LABELS,canonical,topic_groups,topic_correction
GRID=[0.,.2,.4,.62]
L2I={v:i for i,v in enumerate(LABELS)}

def fit_scores(train,valid):
    assert not set(train.prompt_id)&set(valid.prompt_id)
    assert not set(train.text.map(canonical))&set(valid.text.map(canonical))
    view=FourViews();x=view.fit_transform(train.text.tolist())
    clf=LinearSVC(C=.5,random_state=42,class_weight='balanced',max_iter=5000).fit(x,train.model.map(L2I))
    assert np.array_equal(clf.classes_,np.arange(18))
    return clf.decision_function(view.transform(valid.text.tolist()))

def scenarios(frame,seed):
    rng=np.random.default_rng(seed);y=frame.model.map(L2I).to_numpy();sparse=[]
    for g in frame.prompt_id.unique():
        ix=np.flatnonzero(frame.prompt_id.to_numpy()==g)
        sparse.extend(rng.choice(ix,min(len(ix),rng.integers(1,5)),replace=False))
    uneven=np.flatnonzero(rng.random(len(frame))<rng.uniform(.15,1,18)[y])
    def ensure(ix):
        return np.unique(np.r_[ix,[rng.choice(np.flatnonzero(y==c)) for c in range(18) if c not in y[ix]]]).astype(int)
    return {'full':np.arange(len(frame)),'sparse':ensure(np.array(sparse,dtype=int)),'uneven':ensure(uneven)}

def evaluate(frame,scores,seed):
    records=[];preds={}
    for scenario,ix in scenarios(frame,seed).items():
        sub=frame.iloc[ix];groups=topic_groups(sub.text.tolist());y=sub.model.map(L2I)
        # In the proxy SVC is the only active member, with its production weight.
        core=1.5*log_softmax(scores[ix],axis=1)
        for alpha in GRID:
            pred=topic_correction(core,scores[ix],groups,alpha,weight=1.5).argmax(1)
            records.append(dict(scenario=scenario,alpha=alpha,score=balanced_accuracy_score(y,pred)))
            preds[scenario,alpha]=(ix,pred)
    return records,preds

def main():
    p=argparse.ArgumentParser();p.add_argument('--train',required=True);p.add_argument('--out',required=True);a=p.parse_args()
    start=time.monotonic();out=Path(a.out);out.mkdir(parents=True,exist_ok=True)
    tr=pd.read_csv(a.train,dtype={'id':str,'prompt_id':str});folds=[];records=[];predictions=[]
    for fold,(it,iv) in enumerate(GroupKFold(5).split(tr,tr.model,tr.prompt_id)):
        dev=tr.iloc[it].reset_index(drop=True);hold=tr.iloc[iv].reset_index(drop=True);inner=[]
        for k,(ia,ib) in enumerate(GroupKFold(3).split(dev,dev.model,dev.prompt_id)):
            s=fit_scores(dev.iloc[ia],dev.iloc[ib]);r,_=evaluate(dev.iloc[ib].reset_index(drop=True),s,27000+fold*100+k);inner+=r
        rank=pd.DataFrame(inner).groupby('alpha').score.mean()
        chosen=min(GRID,key=lambda x:(-rank[x],x))
        print('outer',fold,'inner-selected alpha',chosen,'inner scores',rank.to_dict(),flush=True)
        scores=fit_scores(dev,hold);r,preds=evaluate(hold,scores,28000+fold)
        records.extend(dict(outer=fold,selected_alpha=chosen,**v) for v in r)
        for scenario in ['full','sparse','uneven']:
            ix,base=preds[scenario,0.];_,new=preds[scenario,chosen]
            frame=hold.iloc[ix][['id','prompt_id','model']].copy();frame['outer']=fold;frame['scenario']=scenario
            frame['baseline']=[LABELS[c] for c in base];frame['candidate']=[LABELS[c] for c in new]
            frame['selected_alpha']=chosen;predictions.append(frame)
        folds.append(dict(fold=fold,selected_alpha=chosen,held_out_ids=hold.id.tolist(),held_out_prompts=hold.prompt_id.unique().tolist(),
                          inner_objective={str(x):float(rank[x]) for x in GRID}))
    predictions=pd.concat(predictions,ignore_index=True);predictions.to_csv(out/'nested_predictions.csv',index=False)
    pd.DataFrame(records).to_csv(out/'outer_grid_diagnostics.csv',index=False)
    summary=[]
    for scenario,sub in predictions.groupby('scenario',sort=False):
        b=balanced_accuracy_score(sub.model,sub.baseline);c=balanced_accuracy_score(sub.model,sub.candidate)
        rng=np.random.default_rng(29001);ixs=[np.flatnonzero(sub.prompt_id.to_numpy()==g) for g in sub.prompt_id.unique()];d=[]
        for _ in range(1000):
            ix=np.concatenate([ixs[k] for k in rng.integers(len(ixs),size=len(ixs))]);s=sub.iloc[ix]
            d.append(balanced_accuracy_score(s.model,s.candidate)-balanced_accuracy_score(s.model,s.baseline))
        summary.append(dict(scenario=scenario,rows=len(sub),baseline=b,candidate=c,delta=c-b,paired_prompt_bootstrap_95ci=np.quantile(d,[.025,.975]).tolist()))
    rank={x:float(np.mean([f['inner_objective'][str(x)] for f in folds])) for x in GRID}
    chosen=min(GRID,key=lambda x:(-rank[x],x))
    result=dict(scope='Nested train-only exact-main-SVC proxy. NOT full 0.651 ensemble OOF or leaderboard evidence.',
                selected_alpha=chosen,grid=GRID,outer_folds=5,inner_folds=3,selection='Mean INNER objectives only, equally weighting full/sparse/uneven scenarios.',
                summary=summary,seconds=time.monotonic()-start,train_sha256=hashlib.sha256(Path(a.train).read_bytes()).hexdigest(),
                external_rows=0,task_trained_assets=0)
    (out/'results.json').write_text(json.dumps(result,indent=2));(out/'fold_manifest.json').write_text(json.dumps(folds,indent=2))
    print(json.dumps(result,indent=2),flush=True)
if __name__=='__main__':main()


In [ ]:
%%writefile "{RUNTIME_DIR}/run_upgrade.py"
"""Offline supervisor for the exact released 0.651 stack and CPU-only checks."""
from pathlib import Path
import argparse,json,os,sys,time,glob,signal
def _terminate(signum,frame):raise SystemExit('Notebook supervisor requested termination')
signal.signal(signal.SIGTERM,_terminate)
os.environ.update(HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',TOKENIZERS_PARALLELISM='false',PYTHONUNBUFFERED='1')
from ntl_upgrade import run_checked,LABELS

def main():
    p=argparse.ArgumentParser();p.add_argument('--mode',choices=['kaggle','preflight','cpu_smoke'],default='kaggle')
    p.add_argument('--data',required=True);p.add_argument('--inputs',default='/kaggle/input');p.add_argument('--out',required=True)
    p.add_argument('--topic_alpha',type=float,default=.62);p.add_argument('--hours',type=float,default=8.5)
    p.add_argument('--started',type=float,default=None);p.add_argument('--validate',type=int,default=0);p.add_argument('--parity_checks',type=int,default=0)
    a=p.parse_args();root=Path(__file__).resolve().parent;out=Path(a.out).resolve();out.mkdir(parents=True,exist_ok=True)
    started=a.started if a.started is not None else time.time();deadline=started+3600*a.hours
    os.environ.update(NTL_DEADLINE=str(deadline),NTL_PARITY_CHECK=str(a.parity_checks),NTL_DROP_Q35='1',NTL_DATA=a.data)
    print('ENHANCED 0.651 RUN START',vars(a),flush=True)
    if a.mode!='preflight':
        for name in ['submission.parquet','submission_baseline_0651.parquet','run_report.json','comp_logit.npy','comp_meta.json']:
            (out/name).unlink(missing_ok=True)
    if a.mode!='cpu_smoke':
        from ntl_inputs import preflight
        report=preflight(a.inputs,out/'preflight_report.json')
        if a.mode=='preflight':print('PREFLIGHT ONLY: model inference was not started.',flush=True);return
    import pandas as pd
    train=pd.read_csv(Path(a.data)/'train.csv',dtype={'id':str,'prompt_id':str})
    test=pd.read_csv(Path(a.data)/'test.csv',dtype={'id':str})
    if not train.model.isin(LABELS).all() or not train.id.is_unique or not test.id.is_unique:raise ValueError('Invalid train/test schema or IDs')
    if a.validate:
        run_checked([sys.executable,'-u',str(root/'validate_upgrade.py'),'--train',str(Path(a.data)/'train.csv'),'--out',str(out/'validation')],deadline-180,out/'validation.log')
    if a.mode=='cpu_smoke':
        cmd=[sys.executable,'-u',str(root/'infer.py'),'--data',a.data,'--models','[]','--sim','0','--svc_w','1.5',
             '--tfidf_w','0','--perpassage','1','--adapt_w','1','--adapt_thr','.15','--post','1','--sinkhorn','1',
             '--topic_alpha',str(a.topic_alpha),'--out',str(out/'held_out_smoke/submission.parquet'),'--cache',str(out/'cache')]
        run_checked(cmd,deadline-120,out/'cpu_smoke.log')
        # Also exercise the real test schema and ID order, using only this CPU head.
        cmd=cmd.copy();i=cmd.index('--sim');del cmd[i:i+2]
        cmd[cmd.index('--out')+1]=str(out/'submission.parquet')
        run_checked(cmd,deadline-120,out/'cpu_sample.log')
        sub=pd.read_parquet(out/'submission.parquet')
        if sub.id.tolist()!=test.id.tolist() or list(sub.columns)!=['id','model']:raise ValueError('CPU sample submission IDs/schema differ')
        result=json.loads((out/'run_report.json').read_text());result.update(mode=a.mode,full_recipe_completed=False,
            scope='CPU-only SVC plumbing check. NOT the neural ensemble or a Kaggle-score prediction.')
        (out/'run_report.json').write_text(json.dumps(result,indent=2))
        print('CPU SMOKE COMPLETE: grouped held-out exercise plus real test IDs; no neural/ensemble/LB claim.',flush=True);return
    import torch
    if torch.cuda.device_count()<2:raise RuntimeError('Select Kaggle GPU T4 x2 before Save & Run All.')
    whl=sorted(glob.glob(str(Path(a.inputs)/'**/bitsandbytes-*.whl'),recursive=True))
    if whl:
        # Preserve offline wheel installation, while exposing its true exit status.
        run_checked([sys.executable,'-m','pip','install','--no-index','--no-deps','-q']+whl,deadline-180,out/'offline_install.log')
    models=report['manifest']['models']
    # Original complementary program's fixed output locations are intentionally kept.
    if out!=Path('/kaggle/working'):raise ValueError('Full scored stack requires output /kaggle/working; CPU mode supports local paths.')
    run_checked([sys.executable,'-u',str(root/'comp_stack.py')],deadline-180,out/'complement.log')
    remaining=deadline-time.time()-180
    if remaining<=0:raise TimeoutError('No main-inference time remains after the complement')
    atlas=report['manifest']['atlas']
    cmd=[sys.executable,'-u',str(root/'infer.py'),'--data',a.data,'--models',json.dumps(models),'--perpassage','1',
         '--tfidf_w','0','--distort_w','0','--svc_w','1.5','--pool',report['manifest']['pool'],
         '--adapt_w','1.0','--adapt_thr','.15','--sinkhorn','1.0','--post','1',
         '--atlas_dir',atlas,'--atlas_base',report['manifest']['atlas_base'],'--atlas_w','.10','--atlas_margin','.3',
         '--atlas_bs','4','--atlas_chunk','3','--atlas_budget',str(remaining-120),
         '--comp_path',str(out/'comp_logit.npy'),'--comp_w','.15','--topic_alpha',str(a.topic_alpha),
         '--out',str(out/'submission.parquet'),'--cache',str(out/'cache')]
    run_checked(cmd,deadline-60,out/'main_inference.log')
    sub=pd.read_parquet(out/'submission.parquet')
    if list(sub.columns)!=['id','model'] or sub.id.tolist()!=test.id.tolist() or not sub.model.isin(LABELS).all():raise ValueError('Final submission verification failed')
    result=json.loads((out/'run_report.json').read_text())
    if not atlas:
        result['full_recipe_completed']=False;result['missing_atlas_bundle']=True
    result.update(notebook_seconds=time.time()-started,mode=a.mode,input_references='Preserved all 31 references from supplied notebook')
    (out/'run_report.json').write_text(json.dumps(result,indent=2))
    print('SUBMISSION VERIFIED',len(sub),'rows; changed rows',result['changed_rows'],'full recipe',result['full_recipe_completed'],flush=True)
    if not result['full_recipe_completed']:print('DEGRADED RUN: ATLAS was not fully mixed. This is not a complete 0.651 recipe.',flush=True)
if __name__=='__main__':main()


This final cell starts the selected mode and verifies the output. Kaggle mode retains the original model ensemble. Logs and a completion report are written beside the submission.

In [ ]:
# Explicitly expose the written runtime to this notebook kernel.
# Writing a helper file does not automatically add its directory to sys.path.
if str(RUNTIME_DIR) not in sys.path:
    sys.path.insert(0, str(RUNTIME_DIR))
from ntl_upgrade import run_checked
command = [sys.executable, "-u", str(RUNTIME_DIR / "run_upgrade.py"),
           "--mode", RUN_MODE, "--data", str(DATA_DIR), "--inputs", str(INPUT_DIR),
           "--out", str(OUTPUT_DIR), "--topic_alpha", str(TOPIC_ALPHA),
           "--hours", str(RUNTIME_LIMIT_HOURS), "--started", str(RUN_STARTED),
           "--validate", str(int(RUN_VALIDATION)), "--parity_checks", str(int(RUN_IN_SAMPLE_CHECKS))]
run_checked(command, RUN_STARTED + RUNTIME_LIMIT_HOURS * 3600 + 15, OUTPUT_DIR / "supervisor.log")
if RUN_MODE != "preflight":
    import json, pandas as pd
    print(json.loads((OUTPUT_DIR / "run_report.json").read_text()))
    display(pd.read_parquet(OUTPUT_DIR / "submission.parquet").head())
